# Занятие 10. Методы машинного обучения

In [ ]:
# 📥 Установка пакетов и библиотек
! pip install matplotlib
! pip install numpy
! pip install scipy
! pip install pandas
! pip install scikit-learn
! pip install torch
! pip install networkx
! pip install plotly

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import copy
import networkx as nx

import sklearn 
import scipy.optimize

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# show()
REPO = "/home/jupyter/project/computational-chemistry-course"
sys.path.append(os.path.join(REPO, "tools"))
from ml_tool import show

# 🔥 Глава 1. Введение

Машинное обучение (machine learning, ML) - раздел искусственного интеллекта, который разрабатывает алгоритмы, способные обучаться на данных и применять найденные закономерности к новым данным, то есть решать задачи без явного программирования. Такие методы опираются на математическую статистику, теорию вероятностей, математический анализ, численные методы, методы оптимизации и теорию графов.

Методы машинного обучения делят на три типа:

1. **Обучение без учителя** 
2. **Обучение с учителем** 
3. **Обучение с подкреплением**

В обучении с учителем для каждого объекта известен правильный ответ. Объект описывается набором чисел - вектором признаков $x$, ответ обозначается $y$. По набору пар $(x_i, y_i)$ нужно построить модель $\hat{y} = f(x)$, которая предсказывает ответ для новых объектов. По типу ответа различают две задачи:

- **Регрессия** - ответ является числом: растворимость, температура кипения, энергия молекулы.
- **Классификация** - ответ является классом из заранее заданного набора: токсично или нет, активно или неактивно.

В обучении без учителя ответов $y$ нет, есть только описания объектов $x$. Его основные задачи - **кластеризация** (поиск групп похожих объектов) и **понижение размерности** (описание объектов меньшим числом признаков).

## 1.1 Библиотека Scikit-learn

**[scikit-learn](https://scikit-learn.org/stable/)** - библиотека для работы с методами машинного обучения в Python. В коде она называется `sklearn`. Библиотека работает с данными в виде таблиц: массивами NumPy и таблицами pandas. В ней собраны методы и инструменты для подготовки данных и оценки качества моделей, а также алгоритмы кластеризации, понижения размерности, регрессии и классификации.

Инструменты библиотеки разложены по модулям:

| Модуль | Что в нём |
|---|---|
| `sklearn.preprocessing` | подготовка признаков: стандартизация, масштабирование |
| `sklearn.model_selection` | разбиение набора данных на выборки |
| `sklearn.cluster` | методы кластеризации |
| `sklearn.decomposition` | понижение размерности: анализ главных компонент и другие разложения |
| `sklearn.linear_model` | линейные модели: линейная и логистическая регрессия |
| `sklearn.svm` | метод опорных векторов |
| `sklearn.ensemble` | ансамбли деревьев решений: случайный лес, градиентный бустинг |
| `sklearn.neural_network` | многослойный перцептрон |
| `sklearn.metrics` | метрики качества моделей и кластеризации |



## 1.2 Библиотека PyTorch 

**PyTorch** - библиотека для построения и обучения нейронных сетей. Данные и параметры моделей хранятся в ней в виде **тензоров** (`tensor`) - многомерных массивов чисел (похожих на `np.array`). Библиотека позволяет автоматически вычислять производные при проведении матричных операций, а также выполнять вычисления на видеокартах. PyTorch достаточно низкоуровневая библиотека, и готовой команды обучения для моделей, как в scikit-learn, нет. Поэтому модель собирают из слоёв, а шаги обучения записывают в коде явно.

Инструменты библиотеки разложены по модулям:

| Модуль | Что в нём |
|---|---|
| `torch` | тензоры и математические операции над ними |
| `torch.autograd` | автоматическое вычисление производных |
| `torch.nn` | слои нейронных сетей, функции активации, сборка модели из слоёв |
| `torch.nn.functional` | функции потерь и функции активации |
| `torch.optim` | оптимизаторы параметров |
| `torch.utils.data` | наборы данных и разбиение на батчи |

Дополнительные модули для PyTorch:
* [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/) - графовые нейронные сети
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) - надстройка, которая берёт на себя цикл обучения
* [torchvision](https://docs.pytorch.org/vision/stable/) - модели и наборы данных для изображений

In [ ]:
# Тензоры: координаты атомов молекулы воды, Å
xyz = torch.tensor([[ 0.000, 0.000, 0.000],          # O
                    [ 0.757, 0.586, 0.000],          # H
                    [-0.757, 0.586, 0.000]])         # H
print("размер тензора:", xyz.shape, " тип чисел:", xyz.dtype)

v1, v2 = xyz[1] - xyz[0], xyz[2] - xyz[0]            # векторы связей O-H: операции записываются как для массивов NumPy
r1, r2 = torch.linalg.norm(v1), torch.linalg.norm(v2)          # длины связей
angle = torch.rad2deg(torch.acos(v1 @ v2 / (r1 * r2)))         # угол H-O-H
print(f"длины связей O-H: {r1.item():.3f} и {r2.item():.3f} Å, угол H-O-H: {angle.item():.1f}°")

xyz_np = xyz.numpy()                                 # тензор -> массив NumPy
xyz_t = torch.tensor(xyz_np)                         # массив NumPy -> тензор
print(type(xyz_np), type(xyz_t))

In [ ]:
# Умножение тензоров: на число, поэлементное и матричное
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[10.0, 20.0],
                  [30.0, 40.0]])
x = torch.tensor([1.0, -1.0])

print("умножение на число, 2 * A:\n", 2 * A)
print("поэлементное умножение, A * B:\n", A * B)                 # каждый элемент A умножается на элемент B с теми же индексами
print("матричное умножение, A @ B:\n", A @ B)                    # строки A умножаются на столбцы B
print("матрица на вектор, A @ x:", A @ x)
print("скалярное произведение векторов, x @ x:", x @ x)

In [ ]:
# Автоматическое вычисление производной: потенциал Морзе связи O-H
D, a, r0 = 460.0, 2.3, 0.96                          # кДж/моль, 1/Å, Å
r = torch.tensor(1.20, requires_grad=True)           # длина связи, Å; requires_grad=True: по этому тензору нужна производная
E = D * (1 - torch.exp(-a * (r - r0))) ** 2          # энергия, кДж/моль
E.backward()                                         # вычисление производной dE/dr, результат записывается в r.grad
print(f"E = {E.item():.1f} кДж/моль, dE/dr = {r.grad.item():.1f} кДж/(моль·Å)")

dE_dr = 2 * D * a * (1 - np.exp(-a * (1.20 - r0))) * np.exp(-a * (1.20 - r0))    # та же производная по формуле
print(f"по формуле: dE/dr = {dE_dr:.1f} кДж/(моль·Å)")

In [ ]:
# Оптимизатор: поиск длины связи с наименьшей энергией градиентным спуском
D, a, r0 = 460.0, 2.3, 0.96                          # кДж/моль, 1/Å, Å
r = torch.tensor(1.20, requires_grad=True)           # начальная длина связи, Å
optimizer = torch.optim.SGD([r], lr=0.0001)          # оптимизатор: что подбирать и длина шага
for step in range(30):
    optimizer.zero_grad()                            # обнуление производной прошлого шага
    E = D * (1 - torch.exp(-a * (r - r0))) ** 2      # энергия при текущей длине связи
    E.backward()                                     # производная dE/dr
    if step % 5 == 0:
        print(f"шаг {step:2d}: r = {r.item():.4f} Å, E = {E.item():6.3f} кДж/моль, dE/dr = {r.grad.item():6.1f} кДж/(моль·Å)")
    optimizer.step()                                 # сдвиг r против производной
print(f"после 30 шагов: r = {r.item():.4f} Å")

## 1.3 Наборы данных и пространство признаков

**Набор данных** (dataset, датасет) - таблица, в которой строка описывает один **объект**, а столбец - один **признак** (feature): число, которым объект характеризуется. Для молекул признаками служат **дескрипторы**. Набор из $n$ объектов с $m$ признаками записывают как **матрицу признаков** $X$ размера $n \times m$, её элемент $x_{ik}$ - значение признака $k$ у объекта $i$.

**Пространство признаков** - векторное пространство, в котором каждому признаку соответствует своя ось. Строка таблицы - вектор $x_i = (x_{i1}, \dots, x_{im})$, то есть точка в пространстве размерности $m$. Похожие объекты лежат в нём близко друг к другу, а близость измеряют **евклидовым расстоянием**:

$$d_{ij} = \sqrt{\sum_{k=1}^{m} (x_{ik} - x_{jk})^2}$$


In [ ]:
# Набор данных: четыре молекулы и два признака
data = pd.DataFrame({
    "Вещество": ["бутан", "пропанол-1", "этандиол-1,2", "метилацетат"],
    "M":    [58.1, 60.1, 62.1, 74.1],                # молярная масса, г/моль
    "TPSA": [0.0, 20.2, 40.5, 26.3],                 # полярная площадь поверхности, Å²
})
X = data[["M", "TPSA"]].to_numpy()                   # матрица признаков: 4 объекта, 2 признака
print("размер матрицы признаков:", X.shape)

i, j = 1, 3                                          # номера двух объектов: пропанол-1 и метилацетат
d = np.sqrt(np.sum((X[i] - X[j]) ** 2))              # евклидово расстояние между ними
print(f"расстояние между объектами {i} и {j}: {d:.1f}")

# Объекты как точки в пространстве признаков
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[:, 0], X[:, 1], color="black", s=40)
for name, (M, TPSA) in zip(data["Вещество"], X):
    ax.annotate(name, (M, TPSA), xytext=(7, -11), textcoords="offset points")
ax.plot(X[[i, j], 0], X[[i, j], 1], color="#1f77b4", lw=1.2, ls="--")      # отрезок между двумя объектами
ax.set_xlim(45, 100)
ax.set_ylim(-8, 47)
ax.set_xlabel("M, г/моль")
ax.set_ylabel("TPSA, Å²")
ax.set_aspect("equal")
plt.show()
data

# ☕︎ Глава 2. Обучение без учителя

**Обучение без учителя** (unsupervised learning) - методы, которые ищут структуру в данных без известных ответов: даны только описания объектов в пространстве признаков, а величины, которую нужно предсказать, нет. Результат - не предсказание, а описание данных: группы похожих объектов или новые координаты, в которых данные проще рассмотреть.

## 2.1 Методы кластеризации 

**Кластеризация** - поиск групп (кластеров) в данных **без заранее известных ответов**: объекты внутри одного кластера должны быть похожи между собой по набору признаков и непохожи на объекты других кластеров.

Похожесть задаётся расстоянием между векторами признаков: чем ближе точки в пространстве признаков, тем более похожи объекты.

In [ ]:
# Набор данных: 24 объекта с двумя признаками A и B
df = pd.DataFrame({
    "A": [1.0, 1.5, 0.8, 1.3, 0.6, 1.7, 5.0, 5.6, 4.6, 5.3, 4.8, 5.9, 8.9, 9.4, 8.5, 9.1, 8.3, 9.7, 5.1, 4.5, 5.7, 4.9, 5.4, 4.3],
    "B": [1.2, 0.8, 1.6, 1.5, 0.9, 1.3, 5.2, 4.7, 5.5, 5.8, 4.5, 5.3, 1.1, 1.6, 0.7, 0.5, 1.4, 1.0, 9.4, 9.9, 9.0, 8.8, 10.1, 9.3],
})

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df["A"], df["B"], color="black", s=40)
ax.set_xlabel("A")
ax.set_ylabel("B")
plt.show()

### 2.1.1 Метод K-средних

**Метод K-средних** (K-means) - алгоритм кластеризации, который делит объекты на заранее заданное число $K$ кластеров так, чтобы сумма квадратов расстояний от объектов до центра своего кластера была минимальной:

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

где $C_k$ - $k$-й кластер, $\mu_k$ - его **центроид**, среднее арифметическое векторов признаков объектов кластера.

Алгоритм итерационный и минимизирует $J$ по шагам:

1. Выбираем $K$ начальных центроидов, обычно случайные объекты набора.
2. Каждый объект относим к ближайшему центроиду.
3. Каждый центроид переносим в среднее объектов своего кластера.
4. Повторяем шаги 2-3, пока центроиды не перестанут двигаться.

Шаги 2 и 3 не могут увеличить $J$, поэтому алгоритм всегда останавливается.

In [ ]:
# Кластеризация K-средних из scikit-learn
N_clusters = 4

km = sklearn.cluster.KMeans(n_clusters=N_clusters, n_init=10, random_state=0)
df["кластер"] = km.fit_predict(df[["A", "B"]])          # номер кластера для каждого объекта

colors = plt.cm.tab10(np.arange(N_clusters) % 10)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df["A"], df["B"], c=colors[df["кластер"]], s=40)
ax.scatter(*km.cluster_centers_.T, c=colors, s=140, marker="X", edgecolor="black", lw=0.8)   # центроиды
ax.set_xlabel("A")
ax.set_ylabel("B")
plt.show()

Алгоритм K-средних не определяет число кластеров, $K$ задаётся заранее. Чтобы выбрать оптимальное число кластеров, кластеризацию повторяют при разных $K$ и сравнивают результаты по одному из критериев.

**Метод локтя** (elbow method). Строят зависимость целевой функции $J$ от $K$. С ростом $K$ она убывает, но после «правильного» числа кластеров убывает медленно: на графике появляется излом - локоть. Критерий нагляден, но субъективен: излом бывает размытым.

In [ ]:
# Метод локтя: целевая функция J при разных K
K_values = range(1, 9)
J = []
for K in K_values:
    km = sklearn.cluster.KMeans(n_clusters=K, 
                                n_init=10,     # число случайных инициализаций центроидов, берётся лучшее решение
                                random_state=0)
    km.fit(df[["A", "B"]])
    J.append(km.inertia_)                                # сумма квадратов расстояний до центроидов

gain = -np.diff(J)                                       # выигрыш в J от добавления ещё одного кластера
K_elbow = K_values[np.argmax(gain[:-1] / gain[1:]) + 1]  # K, после которого выигрыш падает сильнее всего

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(K_values, J, "o-", color="black")
ax1.axvline(K_elbow, color="grey", ls="--")              # найденный локоть
ax1.set_xlabel("число кластеров K")
ax1.set_ylabel("J")
ax2.bar(K_values[1:], gain, color="black")               # столбец при K - выигрыш от K-го кластера
ax2.set_xlabel("число кластеров K")
ax2.set_ylabel("выигрыш в J")
plt.tight_layout()
plt.show()

print("Оптимальное число кластеров по методу локтя: K =", K_elbow)

**Силуэтный коэффициент** (silhouette score). Для каждого объекта $i$ считают

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\ b(i))}$$

где $a(i)$ - среднее расстояние до объектов своего кластера, $b(i)$ - среднее расстояние до объектов ближайшего чужого кластера. Значения лежат в пределах от $-1$ до $1$:

- $s \approx 1$ - объект глубоко внутри своего кластера;
- $s \approx 0$ - объект на границе двух кластеров;
- $s < 0$ - объект, скорее всего, попал не в свой кластер.

Оптимальным считают $K$, при котором средний по всем объектам коэффициент максимален. Критерий учитывает и компактность кластеров, и расстояние между ними.

In [ ]:
# Силуэтный коэффициент при разных K
K_values = range(2, 9)                                   # при K = 1 коэффициент не определён
S = []
for K in K_values:
    km = sklearn.cluster.KMeans(n_clusters=K, n_init=10, random_state=0)
    labels = km.fit_predict(df[["A", "B"]])
    S.append(sklearn.metrics.silhouette_score(df[["A", "B"]], labels))   # средний силуэтный коэффициент

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(K_values, S, "o-", color="black")
ax.set_xlabel("число кластеров K")
ax.set_ylabel("силуэтный коэффициент")
plt.show()

print("Максимум силуэтного коэффициента при K =", K_values[np.argmax(S)])

### 2.1.2 Кластеризация в многомерном пространстве и стандартизация данных

Но признаков у объекта может быть больше двух. Каждый объект - точка в пространстве, размерность которого равна числу признаков, а расстояние между объектами $i$ и $j$ считается по всем $m$ признакам:

$$d_{ij} = \sqrt{\sum_{k=1}^{m} (x_{ik} - x_{jk})^2}$$

Алгоритм K-средних при этом не меняется. Возьмём 16 молекул четырёх классов (алканы, спирты, диолы, сложные эфиры) и опишем каждую четырьмя признаками. 

In [ ]:
# Набор данных: 16 молекул четырёх классов
mols = pd.DataFrame({
    "Вещество": ["бутан", "гексан", "октан", "декан",
                 "пропанол-1", "пентанол-1", "гептанол-1", "нонанол-1",
                 "этандиол-1,2", "бутандиол-1,4", "гександиол-1,6", "октандиол-1,8",
                 "метилацетат", "метилбутаноат", "метилгексаноат", "метилоктаноат"],
    "M":    [58.1, 86.2, 114.2, 142.3, 60.1, 88.1, 116.2, 144.3, 62.1, 90.1, 118.2, 146.2, 74.1, 102.1, 130.2, 158.2],   # молярная масса, г/моль
    "TPSA": [0.0, 0.0, 0.0, 0.0, 20.2, 20.2, 20.2, 20.2, 40.5, 40.5, 40.5, 40.5, 26.3, 26.3, 26.3, 26.3],               # полярная площадь поверхности, Å²
    "HBD":  [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 0, 0, 0, 0],                                                           # число доноров водородной связи
    "HBA":  [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],                                                           # число акцепторов водородной связи
})
mols

In [ ]:
# K-средних на исходных признаках и кластеры в двух трёхмерных проекциях
features = ["M", "TPSA", "HBD", "HBA"]
km = sklearn.cluster.KMeans(n_clusters=4, n_init=10, random_state=0)
mols["кластер"] = km.fit_predict(mols[features])

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]                 # цвет кластера
views = [("M", "TPSA", "HBD"), ("TPSA", "HBD", "HBA")]                # признаки по осям двух графиков
rng = np.random.default_rng(0)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=[", ".join(v) for v in views])
for col, axes in enumerate(views, start=1):
    # небольшой случайный сдвиг, чтобы совпадающие точки не закрывали друг друга
    x, y, z = [mols[a] + rng.uniform(-0.03, 0.03, len(mols)) * (mols[a].max() - mols[a].min()) for a in axes]
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="markers", text=mols["Вещество"], hovertemplate="%{text}<extra></extra>",
                               marker=dict(size=6, color=[colors[k] for k in mols["кластер"]])), row=1, col=col)
    fig.update_scenes(xaxis_title=axes[0], yaxis_title=axes[1], zaxis_title=axes[2], row=1, col=col)
fig.update_layout(width=1000, height=450, showlegend=False, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
# Состав кластеров
pd.set_option("display.max_colwidth", None)  
summary = pd.DataFrame({
    "Вещества":  mols.groupby("кластер")["Вещество"].apply(", ".join),
})
summary

Признаки измерены в разных единицах и различаются по масштабу: молярная масса меняется от 58 до 158 г/моль, а число доноров водородной связи - от 0 до 2. Поэтому расстояние между молекулами определяется молярной массой и TPSA, а числа доноров и акцепторов в него почти не входят. 

**Стандартизация** - преобразование, которое приводит все признаки к одному масштабу: из каждого столбца вычитают его среднее $\bar{x}_k$ и делят на стандартное отклонение $\sigma_k$:

$$x'_{ik} = \frac{x_{ik} - \bar{x}_k}{\sigma_k}$$

После стандартизации каждый признак имеет среднее 0 и стандартное отклонение 1 и вносит в расстояние сопоставимый вклад. Повторим кластеризацию на стандартизованных признаках. 

In [ ]:
# Стандартизация признаков через scikit-learn
features = ["M", "TPSA", "HBD", "HBA"]
scaler = sklearn.preprocessing.StandardScaler()
mols_std = mols[["Вещество"] + features].copy()
mols_std[features] = scaler.fit_transform(mols[features]) 

X = mols[features].to_numpy()
print("среднее:  ", X.mean(axis=0).round(2))
print("ст. откл.:", X.std(axis=0).round(2))
mols_std.round(2)

In [ ]:
# K-средних на стандартизованных признаках и кластеры в двух трёхмерных проекциях
km = sklearn.cluster.KMeans(n_clusters=4, n_init=10, random_state=0)
mols_std["кластер"] = km.fit_predict(mols_std[features])

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]                 # цвет кластера
views = [("M", "TPSA", "HBD"), ("TPSA", "HBD", "HBA")]                # признаки по осям двух графиков
rng = np.random.default_rng(0)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=[", ".join(v) for v in views])
for col, axes in enumerate(views, start=1):
    # небольшой случайный сдвиг, чтобы совпадающие точки не закрывали друг друга
    x, y, z = [mols_std[a] + rng.uniform(-0.03, 0.03, len(mols_std)) * (mols_std[a].max() - mols_std[a].min()) for a in axes]
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="markers", text=mols_std["Вещество"], hovertemplate="%{text}<extra></extra>",
                               marker=dict(size=6, color=[colors[k] for k in mols_std["кластер"]])), row=1, col=col)
    fig.update_scenes(xaxis_title=axes[0], yaxis_title=axes[1], zaxis_title=axes[2], row=1, col=col)
fig.update_layout(width=1100, height=500, showlegend=False, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
# Состав кластеров после стандартизации признаков
pd.set_option("display.max_colwidth", None)             # показывать длинные строки целиком
summary = pd.DataFrame({
    "Вещества":  mols_std.groupby("кластер")["Вещество"].apply(", ".join),
})
summary

## 2.2 Методы понижения размерности

**Понижение размерности** (dimensionality reduction) - переход от исходных $m$ признаков к небольшому числу новых координат, которые сохраняют основную информацию о различиях между объектами.

В предыдущем примере у молекулы было четыре признака, и чтобы увидеть кластеры, пришлось строить две трёхмерные проекции, ни одна из которых не показывает все признаки сразу. В реальных задачах дескрипторов десятки и сотни, и возникают три трудности:

- данные нельзя изобразить в двух- или трёхмерном пространстве
- признаки коррелируют между собой и повторяют одну и ту же информацию: молярная масса, число атомов и объём молекулы растут вместе
- чем больше признаков, тем меньше различаются расстояния между объектами, и методы, основанные на расстояниях, работают хуже

### 2.2.1 Сингулярное разложение

**Сингулярное разложение** (singular value decomposition, SVD) - представление матрицы данных $X$ размера $n \times m$ в виде произведения трёх матриц:

$$\underset{n \times m}{X} \;=\; \underset{n \times r}{U} \;\; \underset{r \times r}{S} \;\; \underset{r \times m}{V^\top}, \qquad r = \min(n, m)$$

- $U$ - **левые сингулярные векторы** $u_i$ (столбцы длины $n$), ортонормированные;
- $S$ - диагональная матрица **сингулярных чисел** $s_1 \geq s_2 \geq \dots \geq s_r \geq 0$;
- $V$ - **правые сингулярные векторы** $v_i$ (столбцы длины $m$), ортонормированный набор направлений в пространстве признаков.

То же разложение можно записать как сумму компонент:

$$X = \sum_{i=1}^{r} s_i \, u_i v_i^\top$$

Сингулярное число $s_i$ показывает вклад $i$-й компоненты в данные. Если оставить в сумме только первые $q$ слагаемых с наибольшими $s_i$, получится наилучшее приближение матрицы $X$ матрицей ранга $q$. 

Разложение можно получить через собственные векторы матрицы $X^\top X$:

1. Строим матрицу $X^\top X$ размера $m \times m$. Она симметрична, её собственные числа неотрицательны.
2. Находим собственные числа $\lambda_1 \geq \lambda_2 \geq \dots \geq 0$ и собственные векторы $v_i$: $X^\top X \, v_i = \lambda_i \, v_i$. Векторы $v_i$ - столбцы матрицы $V$.
3. Сингулярные числа - корни из собственных чисел: $s_i = \sqrt{\lambda_i}$.
4. Левые сингулярные векторы получаем умножением $X$ на $v_i$: $u_i = X \, v_i / s_i$.

На практике разложение считают готовой функцией `np.linalg.svd`.

In [ ]:
# Спектры чистых веществ A, B и C и спектры их смеси в ходе реакции A → B → C
rng = np.random.default_rng(0)
wavelengths = np.linspace(400, 700, 301)            # нм, шаг 1 нм
times = np.linspace(0, 25, 101)                     # мин, шаг 0.25 мин

# Спектры чистых веществ: каждый спектр - сумма двух-трёх гауссовых полос
def band(center, width, height):
    return height * np.exp(-0.5 * ((wavelengths - center) / width) ** 2)

E_A = band(455, 16, 1.0) + band(495, 20, 0.55)                         # вещество A: две полосы
E_B = band(505, 18, 0.5) + band(545, 20, 1.0) + band(590, 18, 0.6)     # вещество B: три полосы
E_C = band(595, 20, 0.6) + band(640, 16, 1.0)                          # вещество C: две полосы

# Концентрации: последовательная реакция первого порядка
k1, k2 = 0.45, 0.25                                 # константы скорости, 1/мин
C_A = np.exp(-k1 * times)
C_B = k1 / (k2 - k1) * (np.exp(-k1 * times) - np.exp(-k2 * times))
C_C = 1 - C_A - C_B

# Спектр смеси в каждый момент времени: спектры веществ, умноженные на концентрации, и шум
X = np.outer(C_A, E_A) + np.outer(C_B, E_B) + np.outer(C_C, E_C)
X = X + 0.01 * rng.normal(size=X.shape)

fig, (ax_s, ax_x) = plt.subplots(1, 2, figsize=(12.5, 4.0))
for name, E, color in [("A", E_A, "#1f77b4"), ("B", E_B, "#ff7f0e"), ("C", E_C, "#2ca02c")]:
    ax_s.plot(wavelengths, E, color=color, lw=1.2)
    ax_s.fill_between(wavelengths, E, color=color, alpha=0.25, lw=0)                     # заливка под кривой
    ax_s.text(wavelengths[E.argmax()], E.max() + 0.03, name, color=color, ha="center")   # подпись над максимумом
ax_s.set_ylim(0, 1.25)
ax_s.set_ylabel("поглощение, усл. ед.")
ax_s.set_title("спектры чистых веществ")

scale = 6                                                         # высота спектра: единица поглощения = 6 мин на оси времени
for i in range(0, len(times), 4):                                 # спектр раз в минуту, от раннего к позднему
    t = times[i]
    color = plt.cm.coolwarm(t / times[-1])                        # синий - начало реакции, красный - конец
    ax_x.fill_between(wavelengths, t, t - scale * X[i], color=color, lw=0)
    ax_x.plot(wavelengths, t - scale * X[i], color="black", lw=0.5)
ax_x.set_ylim(26, -7.5)                                           # время идёт сверху вниз
ax_x.set_yticks([0, 5, 10, 15, 20, 25])
ax_x.set_ylabel("время, мин")
ax_x.set_title("спектры смеси в ходе реакции A → B → C")
for ax in (ax_s, ax_x):
    ax.set_xlim(400, 700)
    ax.set_xlabel("длина волны, нм")
plt.tight_layout()
plt.show()

In [ ]:
# Сингулярное разложение матрицы спектров X
U, s, Vt = np.linalg.svd(X, full_matrices=False)
S = np.diag(s)
print("X:", X.shape, " U:", U.shape, " S:", S.shape, " Vᵀ:", Vt.shape)

# Первые q компонент: столбцы U, сингулярные числа и строки Vᵀ
q = 3
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]                 # цвет компоненты
fig, (ax_u, ax_s, ax_v) = plt.subplots(1, 3, figsize=(12.5, 3.8), gridspec_kw={"width_ratios": [1.4, 1, 1.4]})
for i in range(q):
    ax_u.scatter(times, U[:, i], s=8, color=colors[i], label=f"$u_{i + 1}$")           # столбец U - зависимость от времени
    ax_v.plot(wavelengths, Vt[i], color=colors[i], lw=1.2, label=f"$v_{i + 1}$")       # строка Vᵀ - зависимость от длины волны
ax_s.plot(np.arange(1, 11), s[:10], lw=0.8, zorder=0)                    # линия между точками, под ними
ax_s.scatter(np.arange(1, 11), s[:10])
ax_u.axhline(0, color="grey", lw=0.6)
ax_v.axhline(0, color="grey", lw=0.6)
ax_u.legend(frameon=False, loc="lower right")
ax_v.legend(frameon=False, loc="upper left")
ax_u.set_xlabel("время, мин")
ax_u.set_title("U")
ax_s.set_xlabel("номер компоненты")
ax_s.set_title("S")
ax_v.set_xlabel("длина волны, нм")
ax_v.set_title("Vᵀ")
plt.tight_layout()
plt.show()

Столбцы $U$ и строки $V^\top$ - ещё не концентрации и не спектры веществ. SVD строит компоненты ортогональными, поэтому каждая из них - смесь вкладов A, B и C с произвольным знаком и масштабом.

Перейти от компонент к веществам помогает модель реакции. Концентрации в реакции $A \xrightarrow{k_1} B \xrightarrow{k_2} C$ описываются уравнениями кинетики:

$$C_A = e^{-k_1 t}, \qquad C_B = \frac{k_1}{k_2 - k_1}\left(e^{-k_1 t} - e^{-k_2 t}\right), \qquad C_C = 1 - C_A - C_B$$

Первые три столбца $U$ должны быть линейными комбинациями этих кривых:

$$U_q \approx C \, T, \qquad q = 3$$

где $C$ - матрица концентраций размера $n \times 3$, $T$ - **матрица перехода** размера $3 \times 3$. Константы $k_1$ и $k_2$ подбирают так, чтобы отклонение $U_q$ от $C \, T$ было наименьшим. После этого разложение записывается через вещества:

$$X \approx U_q S_q V_q^\top = C \, (T S_q V_q^\top) = C E^\top$$

Строки матрицы $E^\top = T S_q V_q^\top$ - спектры чистых веществ.

У подгонки есть неоднозначность: пары констант $(k_1, k_2)$ и $(k_2, k_1)$ описывают $U$ одинаково хорошо, потому что кривые в обоих случаях составлены из одних и тех же экспонент $e^{-k_1 t}$ и $e^{-k_2 t}$. Выбрать пару помогает физический смысл: поглощение не может быть отрицательным.

In [ ]:
# Подгонка U(t) уравнениями кинетики реакции A → B → C и переход от компонент SVD к веществам
q = 3                                               # число значимых компонент
U_q, S_q, Vt_q = U[:, :q], S[:q, :q], Vt[:q]

# Концентрации веществ по уравнениям кинетики
def kinetics(t, k1, k2):
    C_A = np.exp(-k1 * t)
    C_B = k1 / (k2 - k1) * (np.exp(-k1 * t) - np.exp(-k2 * t))
    C_C = 1 - C_A - C_B
    return np.column_stack([C_A, C_B, C_C])

# Отклонение столбцов U от линейных комбинаций кинетических кривых: U_q ≈ C T
def residual(params):
    C = kinetics(times, *params)
    T = np.linalg.lstsq(C, U_q, rcond=None)[0]      # матрица перехода T (3 × 3) методом наименьших квадратов
    return (U_q - C @ T).ravel()

colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.4), sharex="col")
for (ax_u, ax_c, ax_e), x0 in zip(axes, [[0.5, 0.1], [0.1, 0.5]]):                 # два начальных приближения для k1 и k2
    fit = scipy.optimize.least_squares(residual, x0=x0)
    C_fit = kinetics(times, *fit.x)                                               # концентрации веществ
    T = np.linalg.lstsq(C_fit, U_q, rcond=None)[0]
    E_fit = (T @ S_q @ Vt_q).T                                                    # спектры веществ: X ≈ C (T S_q V_qᵀ)
    print(f"k1 = {fit.x[0]:.3f}, k2 = {fit.x[1]:.3f} 1/мин: сумма квадратов отклонений {2 * fit.cost:.2e}, "
          f"минимум в спектрах {E_fit.min():.2f}")

    for i, (name, C_true, E_true) in enumerate(zip("ABC", [C_A, C_B, C_C], [E_A, E_B, E_C])):
        ax_u.scatter(times, U_q[:, i], s=8, color=colors[i], label=f"$u_{i + 1}$")
        ax_u.plot(times, (C_fit @ T)[:, i], color="black", lw=0.8)                 # подгонка
        ax_c.plot(times, C_fit[:, i], color=colors[i], lw=1.2, label=name)
        ax_c.plot(times, C_true, color="black", lw=0.8, ls="--")                   # заданные концентрации
        ax_e.plot(wavelengths, E_fit[:, i], color=colors[i], lw=1.2, label=name)
        ax_e.plot(wavelengths, E_true, color="black", lw=0.8, ls="--")             # заданные спектры
    ax_e.axhline(0, color="grey", lw=0.6)
    ax_u.set_ylabel(f"k$_1$ = {fit.x[0]:.2f}, k$_2$ = {fit.x[1]:.2f} 1/мин")
axes[0, 0].set_title("U и подгонка")
axes[0, 1].set_title("концентрации веществ")
axes[0, 2].set_title("спектры веществ")
axes[0, 0].legend(frameon=False, loc="lower right")
axes[0, 1].legend(frameon=False, loc="center right")
axes[0, 2].legend(frameon=False, loc="upper left")
axes[1, 0].set_xlabel("время, мин")
axes[1, 1].set_xlabel("время, мин")
axes[1, 2].set_xlabel("длина волны, нм")
plt.tight_layout()
plt.show()

### 2.2.2 Анализ главных компонент

**Анализ главных компонент** (principal component analysis, PCA) - сингулярное разложение матрицы данных, у которой из каждого столбца вычтено его среднее значение:

$$\tilde{X} = X - \bar{x}, \qquad \tilde{X} = U S V^\top$$

Здесь $\bar{x}$ - строка средних значений признаков. После вычитания центр облака точек находится в начале координат, и разложение описывает разброс данных вокруг центра, а не положение самого центра.

Величины из SVD получают в PCA свои названия:

- векторы $v_i$ - **главные компоненты** (principal components). Это ортогональные направления в пространстве признаков: вдоль $v_1$ разброс данных наибольший, вдоль $v_2$ - наибольший среди направлений, ортогональных $v_1$, и так далее;
- $Z = \tilde{X} V = U S$ - **проекции** объектов на главные компоненты (scores), то есть координаты объектов в новых осях;
- $s_i^2 / (n - 1)$ - дисперсия данных вдоль $i$-й главной компоненты. **Доля объяснённой дисперсии** (explained variance ratio) $s_i^2 / \sum_j s_j^2$ показывает, какую часть общего разброса передаёт эта компонента.

Понижение размерности состоит в том, чтобы оставить первые $q$ столбцов $Z$: каждый объект вместо $m$ признаков описывается $q$ координатами, и при этом сохраняется наибольшая возможная доля дисперсии.

Для центрированных данных $\frac{1}{n-1}\tilde{X}^\top \tilde{X}$ - **ковариационная матрица** признаков. Поэтому главные компоненты - её собственные векторы, а дисперсии вдоль них - её собственные числа. Это те же четыре шага через $X^\top X$ из раздела 2.2.1.

Если признаки измерены в разных единицах, перед PCA данные стандартизуют, как в разделе 2.1.2. Иначе первая главная компонента будет почти совпадать с признаком, у которого разброс значений наибольший.

In [ ]:
# PCA набора молекул из раздела 2.1.2 и K-средних по трём главным компонентам
features = ["M", "TPSA", "HBD", "HBA"]
X_std = sklearn.preprocessing.StandardScaler().fit_transform(mols[features])                 # у каждого признака среднее 0 и ст. отклонение 1
pca = sklearn.decomposition.PCA(n_components=3)
Z = pca.fit_transform(X_std)                                                                 # проекции молекул на три главные компоненты
labels = sklearn.cluster.KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(Z)      # номер кластера каждой молекулы

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]                                        # цвет кластера
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "scene"}]], column_widths=[0.35, 0.65],
                    subplot_titles=["доля объяснённой дисперсии", "проекции на три главные компоненты"])
fig.add_trace(go.Scatter(x=[1, 2, 3], y=pca.explained_variance_ratio_, mode="lines+markers", line=dict(color="grey", width=1),
                         marker=dict(size=9, color="black"), showlegend=False), row=1, col=1)
for c in range(4):                                                                           # отдельный набор точек на каждый кластер - строка легенды
    rows = labels == c
    fig.add_trace(go.Scatter3d(x=Z[rows, 0], y=Z[rows, 1], z=Z[rows, 2], mode="markers", name=f"кластер {c}",
                               text=mols["Вещество"][rows], hovertemplate="%{text}<extra></extra>",
                               marker=dict(size=6, color=colors[c])), row=1, col=2)
fig.update_xaxes(title_text="номер главной компоненты", dtick=1, row=1, col=1)
fig.update_scenes(xaxis_title="z1", yaxis_title="z2", zaxis_title="z3")
fig.update_layout(width=1100, height=500, margin=dict(l=60, r=0, t=40, b=50), legend=dict(itemsizing="constant"))
fig.show()

### 2.2.3 Нелинейные методы понижения размерности

**Нелинейные методы понижения размерности** - методы, в которых новые координаты не являются линейными комбинациями исходных признаков. PCA проецирует данные на плоскость и сохраняет наибольший разброс. Нелинейные методы сохраняют соседство: объекты, близкие в исходном пространстве, остаются близкими на карте, даже если данные лежат на изогнутой поверхности.

| | PCA | t-SNE, UMAP |
|---|---|---|
| Новые координаты | линейные комбинации признаков | формулой через признаки не выражаются |
| Что сохраняется | общая форма облака точек | соседство близких объектов |
| Оси | имеют смысл: состав главных компонент | смысла не имеют |
| Результат | воспроизводимый | зависит от параметров и случайного старта |

- **t-SNE** (t-distributed stochastic neighbor embedding) - подбирает положение точек на карте так, чтобы у каждой сохранились её ближайшие соседи. Главный параметр - **перплексия** (perplexity), примерное число соседей у каждой точки.
- **UMAP** (uniform manifold approximation and projection) - строит граф ближайших соседей и раскладывает его на плоскости. Работает быстрее t-SNE и лучше передаёт взаимное расположение групп.

# 🕸️ Глава 3. Обучение с учителем: нейронные сети

**Обучение с учителем** (supervised learning) - построение модели $\hat{y} = f(x)$ по набору объектов, для которых известны правильные ответы $y$. Набор пар $(x_i, y_i)$ называют **обучающей выборкой** (training set).

У модели есть **параметры**, которые подбираются так, чтобы предсказания $\hat{y}_i$ были как можно ближе к известным ответам $y_i$. Меру их различия называют **функцией потерь** (loss function), а подбор параметров - **обучением** модели.

Цель обучения - не воспроизвести обучающую выборку, а верно предсказывать ответы для новых объектов.

## 3.1 Постановка задачи

В задаче обучения с учителем заданы:

- **матрица признаков** $X$ размера $n \times m$: строка - объект, столбец - признак;
- **вектор ответов** $y$ длины $n$: известный ответ для каждого объекта.

Выбирается **модель** - функция определённого вида, которая по признакам объекта $x$ вычисляет **предсказание** $\hat{y}$. Параметры модели - **веса** $w$ и **смещения** $b$. Выбирается и **функция потерь** - число, которое показывает, насколько предсказания отличаются от ответов. В задачах регрессии это обычно **среднеквадратичная ошибка** (mean squared error, MSE):

$$\mathrm{MSE} = \frac{1}{n} \sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)^2$$

Обучить модель - значит найти веса и смещения, при которых MSE минимальна.

## 3.2 Устройство нейрона 

Простейшая модель - **линейная**: предсказание равно сумме признаков, умноженных на веса $w_k$, плюс смещение $b$:

$$\hat{y} = \sum_{k=1}^{m} w_k x_k + b$$

Эту же модель можно представить как **нейрон** - элемент, из множества которых строятся нейронные сети. Нейрон получает признаки $x_k$, складывает их с весами и применяет к сумме **функцию активации**. У линейной модели функция активации линейная: предсказание равно самой сумме.

In [ ]:
# Набор данных из занятия 6: pKa сопряжённых кислот аммиака и метиламинов, расчёт и эксперимент
amines = pd.DataFrame({
    "Основание":  ["NH3", "CH3NH2", "(CH3)2NH", "(CH3)3N"],
    "pKa_расчёт": [10.22, 11.56, 11.70, 10.41],     # B3LYP/def2-SVP, CPCM(вода), абсолютная схема из занятия 6
    "pKa_эксп":   [9.25, 10.66, 10.73, 9.80],       # CRC Handbook
})
amines

In [ ]:
# Модель ŷ = wx в scikit-learn: экспериментальный pKa по расчётному
X = amines[["pKa_расчёт"]].to_numpy()               # матрица признаков, 4 × 1
y = amines["pKa_эксп"].to_numpy()                   # вектор ответов

model = sklearn.linear_model.LinearRegression(fit_intercept=False)     # без смещения: ŷ = wx
model.fit(X, y)
w = model.coef_[0]                                  # вес
y_pred = model.predict(X)                           # предсказания
print(f"w = {w:.3f}, MSE = {np.mean((y - y_pred) ** 2):.3f}, R² = {model.score(X, y):.3f}")

x_line = np.array([10.0, 12.0])
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[:, 0], y, color="black", s=40)
ax.plot(x_line, w * x_line, color="#1f77b4")
ax.set_xlabel("pKa, расчёт")
ax.set_ylabel("pKa, эксперимент")
plt.show()

### 3.2.1 Функции активации

**Функция активации** (activation function) - функция $\varphi$, которую нейрон применяет к взвешенной сумме признаков:

$$z = \sum_{k=1}^{m} w_k x_k + b, \qquad \hat{y} = \varphi(z)$$

У линейной модели $\varphi(z) = z$. В нейронных сетях используют нелинейные функции активации. Без них модель из любого числа нейронов остаётся линейной моделью: линейная функция от линейных функций - снова линейная. 

| Функция | Формула | Область значений | 
|---|---|---|
| Линейная | $\varphi(z) = z$ | $(-\infty, +\infty)$ |
| Сигмоида (sigmoid) | $\varphi(z) = \dfrac{1}{1 + e^{-z}}$ | $(0, 1)$ |
| Гиперболический тангенс (tanh) | $\varphi(z) = \tanh z$ | $(-1, 1)$ |
| ReLU (rectified linear unit) | $\varphi(z) = \max(0, z)$ | $[0, +\infty)$ |
| Leaky ReLU | $\varphi(z) = \max(\alpha z, z)$ | $(-\infty, +\infty)$ |

Обязательным условием для выбора функции активации является её дифференцируемость: при обучении градиентным спуском нужны производные MSE по весам и смещениям, а в них входит производная $\varphi$.

In [ ]:
# Функции активации: линейная, сигмоида, гиперболический тангенс, ReLU и Leaky ReLU
z = np.linspace(-5, 5, 200)
functions = {
    "линейная": z,
    "сигмоида": 1 / (1 + np.exp(-z)),
    "tanh": np.tanh(z),
    "ReLU": np.maximum(0, z),
    "Leaky ReLU, α = 0.1": np.maximum(0.1 * z, z),
}
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig, axes = plt.subplots(1, 5, figsize=(12.5, 2.8))
for ax, (name, values), color in zip(axes, functions.items(), colors):
    ax.axhline(0, color="grey", lw=0.6)
    ax.axvline(0, color="grey", lw=0.6)
    ax.plot(z, values, color=color, lw=1.5)
    ax.set_title(name)
    ax.set_xlabel("z")
axes[0].set_ylabel("φ(z)")
plt.tight_layout()
plt.show()

## 3.3 Нейронная сеть прямого распространения

**Нейронная сеть прямого распространения** (feedforward neural network) состоит из **слоёв** нейронов. Нейроны каждого слоя получают то, что вычислил предыдущий слой, и передают результат следующему:

| Слой | Что делает |
|---|---|
| **Входной** | принимает признаки объекта $x_1, \dots, x_m$ |
| **Скрытые** | один или несколько слоёв нейронов с нелинейной функцией активации |
| **Выходной** | вычисляет предсказание $\hat{y}$ |

Модель с одним скрытым слоем из $N$ нейронов сначала вычисляет **выходы скрытого слоя**:

$$h_j = \varphi\!\left(\sum_{k=1}^{m} w^{(1)}_{kj}\, x_k + b^{(1)}_j\right), \qquad j = 1, \dots, N$$

а затем предсказание - сумму этих выходов с весами:

$$\hat{y} = \sum_{j=1}^{N} w^{(2)}_j h_j + b^{(2)}$$

Верхний индекс - номер слоя, к которому относятся веса и смещения. Вычисление предсказания по признакам слой за слоем называют **прямым проходом** (forward pass). Слой, в котором каждый нейрон связан со всеми нейронами предыдущего слоя, называют полносвязным (fully connected), а модель из таких слоёв - многослойным перцептроном (multilayer perceptron, MLP).

### 3.3.1 Компьютерное представление модели

В PyTorch модель собирают из слоёв с помощью `torch.nn.Sequential`: слои перечисляются в том порядке, в котором через них проходят данные. При создании модели веса и смещения задаются случайными числами. Чтобы при каждом запуске они были одинаковыми, перед созданием модели вызывают `torch.manual_seed`.

In [ ]:
# Модель 2-2-1 в PyTorch
x = torch.tensor([2.0, 3.0])                         # вход: два признака объекта
W1 = torch.tensor([[0.5, -1.0],                      # веса скрытого слоя, 2 × 2
                   [0.3,  0.7]])
b1 = torch.tensor([0.1, 0.2])                        # смещения скрытых нейронов
w2 = torch.tensor([1.5, 0.8])                        # веса выходного нейрона: от двух скрытых нейронов к выходу
b2 = torch.tensor(-0.3)                              # смещение выходного нейрона

net = torch.nn.Sequential(
    torch.nn.Linear(2, 2),                           # скрытый слой: 2 нейрона
    torch.nn.Identity(),                             # функция активации скрытого слоя: линейная
    torch.nn.Linear(2, 1))                           # выходной слой: предсказание ŷ

with torch.no_grad():                                # веса задаются вручную, производные для этого не нужны
    net[0].weight.copy_(W1.T) # слой Linear хранит веса как «нейроны × входные данные», поэтому W1 транспонируется
    net[0].bias.copy_(b1)
    net[2].weight.copy_(w2.reshape(1, 2))
    net[2].bias.copy_(b2)

y_pred = net(x)                                      # прямой проход: модель вызывается как функция
print(f"ŷ = {y_pred.item():.4f}")

show(net, input=x, verbose=True)

In [ ]:
# Модель 2-2-1 на тензорах
h = x @ W1 + b1                       # взвешенные суммы скрытых нейронов
y_pred = h @ w2 + b2                  # предсказание

print("x =", x)
print("h =", h)
print("ŷ =", y_pred)

### 3.3.2 Предсказание энергии системы двух координат

In [ ]:
# Набор данных: энергия двух связей O-H в зависимости от их длин (модельная ППЭ)
D, a, r0, k12 = 460.0, 2.3, 0.96, -100.0            # кДж/моль, 1/Å, Å, кДж/(моль·Å²): модельные параметры

def energy(r1, r2):
    morse_1 = D * (1 - np.exp(-a * (r1 - r0))) ** 2  # потенциал Морзе первой связи
    morse_2 = D * (1 - np.exp(-a * (r2 - r0))) ** 2  # потенциал Морзе второй связи
    return morse_1 + morse_2 + k12 * (r1 - r0) * (r2 - r0)

rng = np.random.default_rng(0)
pes = pd.DataFrame({"r1": rng.uniform(0.75, 1.8, 400),      # длины связей, Å: 400 случайных точек
                    "r2": rng.uniform(0.75, 1.8, 400)})
pes["E"] = energy(pes["r1"], pes["r2"])                     # энергия, кДж/моль

# Точная поверхность и точки набора данных
r_grid = np.linspace(0.75, 1.8, 100)
R1, R2 = np.meshgrid(r_grid, r_grid)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
im = ax.contourf(R1, R2, energy(R1, R2), levels=20, cmap="RdYlBu_r")
ax.scatter(pes["r1"], pes["r2"], color="black", s=3)
fig.colorbar(im, ax=ax, label="E, кДж/моль")
ax.set_xlabel("r$_1$, Å")
ax.set_ylabel("r$_2$, Å")
ax.set_aspect("equal")
plt.show()
pes

In [ ]:
# Модель в PyTorch
torch.manual_seed(0)                                 # начальные веса воспроизводимы
net = torch.nn.Sequential(
    torch.nn.Linear(2, 8),                           # скрытый слой: 2 признака, 8 нейронов
    torch.nn.Identity(),                             # функция активации скрытого слоя: линейная
    torch.nn.Linear(8, 1))                           # выходной слой: предсказание ŷ
show(net)

## 3.4 Обучение модели

**Обучение модели** - подбор весов и смещений, при которых MSE минимальна. Чтобы обученной модели можно было доверять, её проверяют на объектах, которые в подборе не участвовали. Для этого набор данных заранее делят на три **выборки**:

| Выборка | Доля | Для чего нужна |
|---|---|---|
| **Обучающая** (training set) | около 70 % | по ней подбирают веса и смещения |
| **Валидационная** (validation set) | около 15 % | по ней выбирают **гиперпараметры** - настройки, которые задаются до обучения: число нейронов, скорость обучения, число шагов обучения |
| **Тестовая** (test set) | около 15 % | используется один раз, в конце, для оценки готовой модели на новых объектах |

Валидационная выборка в подборе весов не участвует, но по ней принимают решения о модели. Поэтому итоговую оценку на ней давать нельзя, для этого отложена тестовая выборка.

Сравнение MSE на обучающей и валидационной выборках показывает, что происходит с моделью:

- **недообучение** (underfitting) - MSE велика на обеих выборках: модель слишком простая;
- **переобучение** (overfitting) - MSE на обучающей выборке мала, а на валидационной велика: модель запомнила обучающие объекты вместе со случайными погрешностями и плохо предсказывает новые.

Признаки стандартизуют по среднему и стандартному отклонению обучающей выборки, и те же значения применяют к двум другим выборкам.

In [ ]:
# Разбиение набора данных на обучающую, валидационную и тестовую выборки: 70 %, 15 % и 15 %
X_all = pes[["r1", "r2"]].to_numpy()                 # Признаки для входного слоя
y_all = pes["E"].to_numpy()                          # Ответы для выходного слоя
r_train, r_rest, y_train, y_rest = sklearn.model_selection.train_test_split(X_all, y_all, test_size=0.3, random_state=0)
r_val, r_test, y_val, y_test = sklearn.model_selection.train_test_split(r_rest, y_rest, test_size=0.5, random_state=0)

# Стандартизация признаков: среднее и стандартное отклонение берутся только из обучающей выборки
scaler = sklearn.preprocessing.StandardScaler().fit(r_train)
X_train, X_val, X_test = scaler.transform(r_train), scaler.transform(r_val), scaler.transform(r_test)
print("обучающая:", X_train.shape, " валидационная:", X_val.shape, " тестовая:", X_test.shape)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0), sharex=True, sharey=True)
for ax, (name, r, color) in zip(axes, [("обучающая", r_train, "#1f77b4"), ("валидационная", r_val, "#ff7f0e"), ("тестовая", r_test, "#2ca02c")]):
    ax.scatter(r[:, 0], r[:, 1], s=10, color=color)
    ax.set_title(f"{name} выборка: {len(r)} точек")
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
plt.show()

### 3.4.1 Обучение модели с линейной функцией активации

Если у всех нейронов функция активации линейная, модель остаётся линейной, и её веса и смещение находятся по формуле метода наименьших квадратов: $w = (X^\top X)^{-1} X^\top y$. Обучение занимает одно действие, но предсказание такой модели - всегда плоскость $\hat{y} = w_1 x_1 + w_2 x_2 + b$.

In [ ]:
# Модель 2-8-1 с линейной функцией активации
torch.manual_seed(0)               # начальные веса воспроизводимы
net_1 = torch.nn.Sequential(
    torch.nn.Linear(2, 8),         # скрытый слой: 2 признака, 8 нейронов
    torch.nn.Identity(),           # функция активации скрытого слоя: линейная
    torch.nn.Linear(8, 1))         # выходной слой: предсказание ŷ

net_2 = copy.deepcopy(net_1)       # Копия модели 

In [ ]:
show(net_1)

In [ ]:
# Обучающая выборка в виде тензоров
X_t = torch.tensor(X_train, dtype=torch.float32)                             # признаки обучающей выборки: тензор 280 × 2
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)                # ответы: тензор 280 × 1

In [ ]:
# Обучение без итераций: веса и смещение выходного нейрона по методу наименьших квадратов
with torch.no_grad():
    H = net_1[1](net_1[0](X_t))                                                  # выходы скрытого слоя: тензор 280 × 8
    H1 = torch.cat([H, torch.ones(len(H), 1)], dim=1)                        # столбец единиц для смещения: тензор 280 × 9
    solution = torch.linalg.lstsq(H1, y_t).solution                          # решение задачи наименьших квадратов: 9 чисел
    net_1[2].weight.copy_(solution[:8].T)                                      # веса выходного нейрона
    net_1[2].bias.copy_(solution[8])                                           # смещение выходного нейрона
    mse = torch.nn.functional.mse_loss(net_1(X_t), y_t).item()                 # MSE на обучающей выборке

In [ ]:
W1, b1 = net_1[0].weight.detach().numpy(), net_1[0].bias.detach().numpy()        # веса и смещения скрытого слоя: 8 × 2 и 8
w2, b2 = net_1[2].weight.detach().numpy()[0], net_1[2].bias.item()               # веса и смещение выходного нейрона
with np.printoptions(precision=2, suppress=True):
    print("веса скрытого слоя:\n", W1)
    print("смещения скрытого слоя:", b1)
    print("веса выходного нейрона:", w2)
    print("смещение выходного нейрона:", round(b2, 2))
print(f"MSE, обучающая выборка: {mse:.0f}")

In [ ]:
show(net_1, verbose=True)

In [ ]:
# Обучение по всей обучающей выборке методом L-BFGS
optimizer = torch.optim.LBFGS(net_2.parameters(),               # Настройки обучения
                              max_iter=1000,                    # Максимальное число итераций
                              )

def closure():                                                  # MSE и её производные по весам и смещениям
    optimizer.zero_grad()
    mse = torch.nn.functional.mse_loss(net_2(X_t), y_t)
    mse.backward()
    return mse

optimizer.step(closure)                                         # подбор весов и смещений
mse = torch.nn.functional.mse_loss(net_2(X_t), y_t).item()      # MSE на обучающей выборке

In [ ]:
show(net_2, verbose=True)

In [ ]:
# Предсказания двух моделей и точная энергия на сетке значений r1 и r2
grid = torch.tensor(scaler.transform(np.column_stack([R1.ravel(), R2.ravel()])), dtype=torch.float32)
with torch.no_grad():
    E_1 = net_1(grid).numpy()[:, 0]                                          # предсказание модели net_1: обучение без итераций
    E_2 = net_2(grid).numpy()[:, 0]                                          # предсказание модели net_2: обучение методом L-BFGS

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), sharey=True, constrained_layout=True)
for ax, title, values in [(axes[0], "модель net_1: без итераций", E_1), (axes[1], "модель net_2: L-BFGS", E_2), (axes[2], "точная энергия", energy(R1, R2).ravel())]:
    im = ax.contourf(R1, R2, values.reshape(R1.shape), levels=np.linspace(-100, 650, 16), cmap="RdYlBu_r")
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
fig.colorbar(im, ax=axes, label="E, кДж/моль")
plt.show()

### 3.4.2 Обучение модели методом градиентного спуска 

**Градиентный спуск** (gradient descent) - итерационный метод поиска минимума функции потерь. На каждом шаге вес сдвигают в сторону убывания MSE, то есть против производной:

$$w \leftarrow w - \eta \, \frac{d\,\mathrm{MSE}}{dw}$$

где $\eta$ - **скорость обучения** (learning rate), она задаёт длину шага. Для модели $\hat{y} = wx$ производная равна

$$\frac{d\,\mathrm{MSE}}{dw} = \frac{2}{n} \sum_{i} (\hat{y}_i - y_i)\, x_i, \qquad \hat{y}_i = w x_i$$

Шаги повторяют, пока MSE не перестанет уменьшаться. Если весов и смещений несколько, вместо производной берут **градиент** - вектор производных MSE по каждому из них, и шаг делается сразу по всем.

В отличие от формулы наименьших квадратов, градиентный спуск подходит и для нелинейных моделей: достаточно уметь вычислять производные MSE по весам и смещениям.

Шаг градиентного спуска можно делать по разному числу объектов. Чем их больше, тем точнее направление шага, но тем дороже сам шаг оптимизации параметров:

| Производные вычисляются | Шагов за проход по выборке из 280 объектов | Что получается |
|---|---|---|
| по всей обучающей выборке | 1 | направление шага точное, но на каждый шаг нужен расчёт по всем объектам; для больших наборов данных это слишком долго |
| по одному объекту | 280 | шаг дешёвый, но подгоняет модель под один объект: направление сильно меняется от шага к шагу, и MSE ведёт себя нестабильно; объекты обрабатываются по одному, поэтому проход по выборке идёт дольше всего |
| по группе из 20 объектов | 14 | компромисс: производные усредняются по группе и меняются от шага к шагу слабо, а вся группа обрабатывается одной матричной операцией |

- **Батч** (batch) - группа объектов, по которой вычисляют производные и делают один шаг. Объекты распределяются по батчам случайно. В PyTorch батчи готовит `torch.utils.data.DataLoader`, размер батча задаёт параметр `batch_size`.

- **Эпоха** (epoch) - один проход по всей обучающей выборке. Если в выборке 280 объектов, а в батче 20, эпоха состоит из 14 шагов. Перед каждой эпохой объекты распределяются по батчам заново (`shuffle=True`).

In [ ]:
# Та же модель 2-8-1, но с сигмоидой в скрытом слое
torch.manual_seed(0)               # начальные веса и порядок батчей воспроизводимы
net = torch.nn.Sequential(
    torch.nn.Linear(2, 8),         # скрытый слой: 2 признака, 8 нейронов
    torch.nn.Sigmoid(),            # функция активации скрытого слоя: сигмоида
    torch.nn.Linear(8, 1))         # выходной слой: предсказание ŷ

In [ ]:
show(net, verbose=True)

In [ ]:
# Предсказание модели со случайными весами (до обучения)
def predict(X):    # предсказания модели для массива NumPy
    with torch.no_grad():
        return net(torch.tensor(X, dtype=torch.float32)).numpy()[:, 0]

grid = scaler.transform(np.column_stack([R1.ravel(), R2.ravel()]))           # сетка в стандартизованных признаках
fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.6), constrained_layout=True)
for ax, title, values, levels in [(axes[0], "модель со случайными весами", predict(grid), 16),
                                  (axes[1], "точная энергия", energy(R1, R2).ravel(), np.linspace(-100, 650, 16))]:
    im = ax.contourf(R1, R2, values.reshape(R1.shape), levels=levels, cmap="RdYlBu_r")
    fig.colorbar(im, ax=ax, label="E, кДж/моль")
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_ylabel("r$_2$, Å")
    ax.set_aspect("equal")
plt.show()

In [ ]:
# Обучающая выборка в виде тензоров и её разбиение на батчи
X_t = torch.tensor(X_train, dtype=torch.float32)                             # признаки обучающей выборки: тензор 280 × 2
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)                # ответы: тензор 280 × 1
batches = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_t, y_t),
                                      batch_size=20, shuffle=True)           # батчи по 20 объектов, порядок объектов случайный

In [ ]:
# Батчи одной эпохи на плоскости r1, r2
state = torch.get_rng_state()                                                # состояние генератора случайных чисел до просмотра батчей
epoch_batches = [scaler.inverse_transform(X_batch.numpy()) for X_batch, y_batch in batches]   # один проход по выборке: длины связей, Å
torch.set_rng_state(state)                                                   # возврат состояния: обучение начнётся с этих же батчей

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), constrained_layout=True)
for ax, k, color in [(axes[0], 0, "#1f77b4"), (axes[1], 1, "#ff7f0e")]:
    ax.scatter(r_train[:, 0], r_train[:, 1], s=8, color="lightgrey")         # вся обучающая выборка
    ax.scatter(epoch_batches[k][:, 0], epoch_batches[k][:, 1], s=30, color=color)  # объекты одного батча
    ax.set_title(f"батч {k + 1}: {len(epoch_batches[k])} объектов")
number = np.concatenate([np.full(len(batch), k + 1) for k, batch in enumerate(epoch_batches)]) # номер батча для каждого объекта
points = np.concatenate(epoch_batches)
im = axes[2].scatter(points[:, 0], points[:, 1], s=16, c=number, cmap="RdYlBu_r", edgecolors="grey", linewidths=0.3)
axes[2].set_title(f"эпоха: все {len(epoch_batches)} батчей")
fig.colorbar(im, ax=axes[2], label="номер батча")
for ax in axes:
    ax.set_xlabel("r$_1$, Å")
    ax.set_ylabel("r$_2$, Å")
    ax.set_aspect("equal")
plt.show()

In [ ]:
# Обучение методом обратного распространения ошибки
optimizer = torch.optim.Adam(net.parameters(), lr=0.05)                      # оптимизатор Adam и скорость обучения
n_epochs = 1000                                                              # число эпох

mse_curve = []
for epoch in range(n_epochs):
    for X_batch, y_batch in batches:                                         # один шаг обучения на каждый батч
        optimizer.zero_grad()                                                # обнуление производных прошлого шага
        mse = torch.nn.functional.mse_loss(net(X_batch), y_batch)            # прямой проход и MSE на батче
        mse.backward()                                                       # обратный проход: производные MSE по весам и смещениям
        optimizer.step()                                                     # обновление весов и смещений
    with torch.no_grad():
        mse_curve.append(torch.nn.functional.mse_loss(net(X_t), y_t).item()) # MSE на всей обучающей выборке после эпохи

In [ ]:
show(net, verbose=True)

In [ ]:
# MSE по эпохам, предсказание модели и точная энергия на сетке значений r1 и r2
grid = scaler.transform(np.column_stack([R1.ravel(), R2.ravel()]))
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), constrained_layout=True)
axes[0].semilogy(mse_curve, color="#1f77b4", lw=0.9)
axes[0].set_xlabel("эпоха")
axes[0].set_ylabel("MSE, обучающая выборка")
for ax, title, values in [(axes[1], "Предсказание модели", predict(grid)), (axes[2], "Точная энергия", energy(R1, R2).ravel())]:
    im = ax.contourf(R1, R2, values.reshape(R1.shape), levels=np.linspace(-100, 650, 16), cmap="RdYlBu_r")
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_ylabel("r$_2$, Å")
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes[1:], label="E, кДж/моль")
plt.show()

### 3.4.3 Метод обратного распространения ошибки

**Метод обратного распространения ошибки** (backpropagation) - способ вычислить производные MSE по всем весам и смещениям модели за один проход от выхода к входу. Он основан на правиле дифференцирования сложной функции: предсказание зависит от весов выходного нейрона напрямую, а от весов скрытого слоя - через выходы скрытых нейронов $h_j$.

Один шаг обучения состоит из трёх действий:

1. **Прямой проход** (forward pass): по признакам вычисляются выходы скрытого слоя $h_j$ и предсказание $\hat{y}$.
2. **Обратный проход** (backward pass): вычисляется ошибка предсказания, а по ней производные MSE - сначала для выходного нейрона, затем для скрытого слоя.
3. **Обновление**: каждый вес и каждое смещение сдвигаются против своей производной, как в градиентном спуске.

Для модели с одним скрытым слоем обратный проход записывается так. Ошибка предсказания для объекта $i$:

$$\delta_i = \hat{y}_i - y_i$$

Производные по весам выходного нейрона:

$$\frac{\partial\,\mathrm{MSE}}{\partial w^{(2)}_j} = \frac{2}{n}\sum_{i} \delta_i \, h_{ij}$$

Производные по весам скрытого слоя. Та же ошибка умножается на вес связи, по которой она возвращается к нейрону $j$, и на производную функции активации:

$$\frac{\partial\,\mathrm{MSE}}{\partial w^{(1)}_{kj}} = \frac{2}{n}\sum_{i} \delta_i \, w^{(2)}_j \, \varphi'(z_{ij}) \, x_{ik}$$

Здесь $x_{ik}$ - признак $k$ объекта $i$, $z_{ij}$ и $h_{ij} = \varphi(z_{ij})$ - взвешенная сумма и выход скрытого нейрона $j$ для этого объекта. Производные по смещениям получаются так же, без множителей $h_{ij}$ и $x_{ik}$.

In [ ]:
# Модель с двумя скрытыми слоями 2-8-8-1: та же сигмоида
torch.manual_seed(0)               # начальные веса и порядок батчей воспроизводимы
net = torch.nn.Sequential(
    torch.nn.Linear(2, 8),         # первый скрытый слой: 2 признака, 8 нейронов
    torch.nn.Sigmoid(),            # функция активации первого скрытого слоя: сигмоида
    torch.nn.Linear(8, 8),         # второй скрытый слой: 8 нейронов
    torch.nn.Sigmoid(),            # функция активации второго скрытого слоя: сигмоида
    torch.nn.Linear(8, 1))         # выходной слой: предсказание ŷ

In [ ]:
show(net, verbose=True)

In [ ]:
# Обучающая выборка в виде тензоров и её разбиение на батчи
X_t = torch.tensor(X_train, dtype=torch.float32)                             # признаки обучающей выборки: тензор 280 × 2
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)                # ответы: тензор 280 × 1
batches = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_t, y_t),
                                      batch_size=20, shuffle=True)           # батчи по 20 объектов, порядок объектов случайный

In [ ]:
# Обучение тем же способом: скорость обучения 0.01, 1500 эпох
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)                      # оптимизатор Adam и скорость обучения
n_epochs = 1500                                                              # число эпох

mse_curve = []
for epoch in range(n_epochs):
    for X_batch, y_batch in batches:                                         # один шаг обучения на каждый батч
        optimizer.zero_grad()                                                # обнуление производных прошлого шага
        mse = torch.nn.functional.mse_loss(net(X_batch), y_batch)            # прямой проход и MSE на батче
        mse.backward()                                                       # обратный проход: производные MSE по весам и смещениям
        optimizer.step()                                                     # обновление весов и смещений
    with torch.no_grad():
        mse_curve.append(torch.nn.functional.mse_loss(net(X_t), y_t).item()) # MSE на всей обучающей выборке после эпохи

In [ ]:
show(net, verbose=True)

In [ ]:
# MSE по эпохам, предсказание модели и точная энергия на сетке значений r1 и r2
grid = scaler.transform(np.column_stack([R1.ravel(), R2.ravel()]))
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), constrained_layout=True)
axes[0].semilogy(mse_curve, color="#1f77b4", lw=0.9)
axes[0].set_xlabel("эпоха")
axes[0].set_ylabel("MSE, обучающая выборка")
for ax, title, values in [(axes[1], "предсказание модели", predict(grid)), (axes[2], "точная энергия", energy(R1, R2).ravel())]:
    im = ax.contourf(R1, R2, values.reshape(R1.shape), levels=np.linspace(-100, 650, 16), cmap="RdYlBu_r")
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_ylabel("r$_2$, Å")
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes[1:], label="E, кДж/моль")
plt.show()

### 3.4.4 Метрики качества модели

**Метрики качества** - числа, которыми оценивают, насколько предсказания модели близки к ответам. Для регрессии основных четыре:

| Метрика | Формула | Единицы | Как читать |
|---|---|---|---|
| **MSE** (mean squared error) | $\frac{1}{n}\sum_i (y_i - \hat{y}_i)^2$ | квадрат единиц ответа | функция потерь, по которой обучалась модель; сильно реагирует на редкие большие отклонения |
| **RMSE** (root mean squared error) | $\sqrt{\mathrm{MSE}}$ | единицы ответа | типичное отклонение предсказания от ответа |
| **MAE** (mean absolute error) | $\frac{1}{n}\sum_i \lvert y_i - \hat{y}_i \rvert$ | единицы ответа | среднее отклонение по модулю; к редким большим отклонениям менее чувствительна |
| **R²** (коэффициент детерминации) | $1 - \dfrac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$ | безразмерный | доля разброса ответов, которую объясняет модель: 1 - идеальное предсказание, 0 - не лучше среднего значения $\bar{y}$ |

Метрики считают отдельно на каждой выборке. Их сравнение показывает, что происходит с моделью:

| RMSE на обучающей выборке | RMSE на валидационной выборке | Что это значит |
|---|---|---|
| большая | большая | **недообучение**: модель слишком простая |
| малая | близка к обучающей | модель подходит к данным |
| малая | намного больше, чем на обучающей | **переобучение**: модель подогнана под обучающие объекты вместе с их погрешностями |

Число скрытых слоёв и число нейронов в слое - гиперпараметры. Их подбирают так: обучают несколько моделей разного размера и выбирают ту, у которой RMSE на валидационной выборке наименьшая.

До сих пор энергии в наборе данных были точными, а объектов было много, поэтому переобучения не возникало. Приблизим задачу к реальной: оставим в обучающей выборке 100 объектов и добавим ко всем энергиям случайную погрешность со стандартным отклонением 20 кДж/моль. 

In [ ]:
# Данные для опыта: 100 объектов обучающей выборки и случайная погрешность в энергиях
sigma = 20.0                                                                 # стандартное отклонение погрешности, кДж/моль
rng = np.random.default_rng(1)
X_small = X_train[:100]                                                      # малая обучающая выборка: 100 объектов из 280
y_small = y_train[:100] + rng.normal(0, sigma, 100)                          # её ответы с погрешностью
y_val_noisy = y_val + rng.normal(0, sigma, len(y_val))                       # ответы валидационной выборки с погрешностью
y_test_noisy = y_test + rng.normal(0, sigma, len(y_test))                    # ответы тестовой выборки с погрешностью
print("обучающая:", X_small.shape, " валидационная:", X_val.shape, " тестовая:", X_test.shape)

In [ ]:
# Функции для опыта: обучение модели заданного размера и RMSE на выборке
y_mean, y_std = y_small.mean(), y_small.std()                                # ответы тоже стандартизуются: так обучение устойчивее
X_t = torch.tensor(X_small, dtype=torch.float32)                             # признаки малой обучающей выборки: тензор 100 × 2
y_t = torch.tensor((y_small - y_mean) / y_std, dtype=torch.float32).unsqueeze(1)             # стандартизованные ответы: тензор 100 × 1

def train(n_layers, n_neurons):                                              # модель: n_layers скрытых слоёв по n_neurons нейронов
    torch.manual_seed(0)                                                     # начальные веса воспроизводимы
    layers, n_in = [], 2
    for layer in range(n_layers):
        layers += [torch.nn.Linear(n_in, n_neurons), torch.nn.Sigmoid()]     # скрытый слой и его функция активации
        n_in = n_neurons
    net = torch.nn.Sequential(*layers, torch.nn.Linear(n_in, 1))             # в конце выходной слой

    optimizer = torch.optim.LBFGS(net.parameters(), max_iter=2000, line_search_fn="strong_wolfe")
    def closure():                                                           # MSE и её производные по весам и смещениям
        optimizer.zero_grad()
        mse = torch.nn.functional.mse_loss(net(X_t), y_t)
        mse.backward()
        return mse
    optimizer.step(closure)                                                  # обучение по всей малой обучающей выборке
    return net

def predict_net(net, X):                                                     # предсказания модели в кДж/моль для массива NumPy
    with torch.no_grad():
        return net(torch.tensor(X, dtype=torch.float32)).numpy()[:, 0] * y_std + y_mean

def rmse(net, X, y):                                                         # RMSE модели на выборке, кДж/моль
    return np.sqrt(np.mean((y - predict_net(net, X)) ** 2))

In [ ]:
# Обучение 21 модели: 1, 2 или 3 скрытых слоя, от 1 до 64 нейронов в слое
nets, rows = {}, []
for n_layers in [1, 2, 3]:
    for n_neurons in [1, 2, 4, 8, 16, 32, 64]:
        net = train(n_layers, n_neurons)
        nets[n_layers, n_neurons] = net                                      # обученные модели сохраняются в словаре
        rows.append({"слоёв": n_layers, "нейронов": n_neurons,
                     "весов и смещений": sum(p.numel() for p in net.parameters()),
                     "RMSE обуч.": rmse(net, X_small, y_small),
                     "RMSE валид.": rmse(net, X_val, y_val_noisy)})
results = pd.DataFrame(rows)
results.pivot(index="слоёв", columns="нейронов", values=["RMSE обуч.", "RMSE валид."]).round(0).astype(int)

In [ ]:
# RMSE на обучающей и валидационной выборках в зависимости от числа нейронов в слое
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), sharey=True)
for ax, n_layers in zip(axes, [1, 2, 3]):
    part = results[results["слоёв"] == n_layers]
    ax.plot(part["нейронов"], part["RMSE обуч."], "o-", color="#1f77b4", lw=1.2, label="обучающая выборка")
    ax.plot(part["нейронов"], part["RMSE валид."], "o-", color="#ff7f0e", lw=1.2, label="валидационная выборка")
    ax.axhline(sigma, color="black", lw=0.8, ls="--", label="погрешность в данных")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(part["нейронов"], part["нейронов"])
    ax.set_title(f"скрытых слоёв: {n_layers}")
    ax.set_xlabel("нейронов в слое")
axes[0].set_ylabel("RMSE, кДж/моль")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Предсказания трёх моделей и точная энергия: недообучение, лучшая модель по валидационной выборке, переобучение
best = results.loc[results["RMSE валид."].idxmin()]                          # строка таблицы с наименьшей RMSE на валидационной выборке
best_net = nets[best["слоёв"], best["нейронов"]]
print(f"лучшая модель: скрытых слоёв {best['слоёв']:.0f}, нейронов в слое {best['нейронов']:.0f}")

grid = scaler.transform(np.column_stack([R1.ravel(), R2.ravel()]))
maps = [("1 слой, 1 нейрон:\nнедообучение", predict_net(nets[1, 1], grid)),
        ("лучшая модель\nпо валидационной выборке", predict_net(best_net, grid)),
        ("2 слоя по 64 нейрона:\nпереобучение", predict_net(nets[2, 64], grid)),
        ("точная энергия\n", energy(R1, R2).ravel())]
fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.4), sharey=True, constrained_layout=True)
for ax, (title, values) in zip(axes, maps):
    im = ax.contourf(R1, R2, values.reshape(R1.shape), levels=np.linspace(-100, 650, 16), cmap="RdYlBu_r", extend="both")
    ax.scatter(r_train[:100, 0], r_train[:100, 1], s=3, color="black")       # объекты малой обучающей выборки
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
fig.colorbar(im, ax=axes, label="E, кДж/моль")
plt.show()

In [ ]:
# Итоговая оценка лучшей модели на тестовой выборке
y_pred = predict_net(best_net, X_test)   # предсказания для тестовой выборки, кДж/моль
rmse_test = np.sqrt(np.mean((y_test_noisy - y_pred) ** 2))
mae_test = np.mean(np.abs(y_test_noisy - y_pred))
r2_test = 1 - np.sum((y_test_noisy - y_pred) ** 2) / np.sum((y_test_noisy - y_test_noisy.mean()) ** 2)
print(f"валидационная выборка: RMSE = {rmse(best_net, X_val, y_val_noisy):.1f} кДж/моль")
print(f"тестовая выборка:      RMSE = {rmse_test:.1f} кДж/моль, MAE = {mae_test:.1f} кДж/моль, R² = {r2_test:.3f}")

fig, ax = plt.subplots(figsize=(4.2, 4))
ax.scatter(y_test_noisy, y_pred, s=12, color="#2ca02c")   # каждая точка - объект тестовой выборки
ax.plot([0, 600], [0, 600], color="black", lw=0.8)          # линия идеального предсказания
ax.set_xlabel("E, кДж/моль")
ax.set_ylabel("предсказание модели, кДж/моль")
ax.set_title("тестовая выборка")
ax.set_aspect("equal")
plt.show()

## 3.5 Классификация

**Классификация** (classification) - задача обучения с учителем, в которой ответ модели - не число, а **класс** объекта. Классы кодируют целыми числами: 0, 1, 2 и так далее.

Нейронная сеть для классификации устроена так же, как для регрессии, но есть отличия:

- в выходном слое столько нейронов, сколько классов
- функция активации выходного слоя - **softmax**. Она превращает выходы нейронов $\hat{y}_k$ в **вероятности** классов, положительные числа с суммой 1:

$$p_k = \frac{e^{\hat{y}_k}}{\sum_l e^{\hat{y}_l}}$$

- предсказанный класс - тот, у которого вероятность наибольшая
- функцией потерь служит не MSE, а **перекрёстная энтропия** (cross-entropy)
- качество оценивают **долей верных предсказаний** (accuracy)

**Перекрёстная энтропия** (cross-entropy) - функция потерь для классификации. Она показывает, какую вероятность модель дала правильному классу.

Для одного объекта она равна

$$-\ln p_\text{прав}$$

где $p_\text{прав}$ - вероятность правильного класса после softmax. Перекрёстная энтропия выборки - среднее по всем её объектам.

| Вероятность правильного класса | $-\ln p_\text{прав}$ | Что это значит |
|---|---|---|
| 1 | 0 | модель уверена и права |
| 0.9 | 0.11 | модель права, но не совсем уверена |
| 0.25 | 1.39 | модель не различает четыре класса |
| 0.1 | 2.30 | модель уверена и ошибается |

In [ ]:
# Набор данных: фазовое состояние смеси бифенил (A) - нафталин (B) при заданных составе и температуре
R = 8.314                  # Дж/(моль·К)

# Температуры и энтальпии плавления: NIST Chemistry WebBook 
Tm_A, dH_A = 342.10, 18576                           # бифенил: температура плавления, К, и энтальпия плавления, Дж/моль
Tm_B, dH_B = 353.38, 19046                           # нафталин

def saturation(T):                                   # состав расплава, насыщенного кристаллами: уравнение Шрёдера, T в °C
    x_A = np.exp(-dH_A / R * (1 / (T + 273.15) - 1 / Tm_A))     # мольная доля A в расплаве над кристаллами A
    x_B = np.exp(-dH_B / R * (1 / (T + 273.15) - 1 / Tm_B))     # мольная доля B в расплаве над кристаллами B
    return x_A, x_B

rng = np.random.default_rng(0)
mix = pd.DataFrame({"x": rng.uniform(0, 1, 400),                # мольная доля нафталина: 400 случайных состояний
                    "T": rng.uniform(20, 90, 400)})             # температура, °C
x_A, x_B = saturation(mix["T"])
mix["класс"] = np.where(x_A + x_B < 1, 3,                       # 3 - кристаллы A и B: ниже температуры эвтектики расплава нет
                        np.where(mix["x"] > x_B, 2,             # 2 - расплав и кристаллы B
                                 np.where(1 - mix["x"] > x_A, 1, 0)))     # 1 - расплав и кристаллы A, 0 - расплав

# Границы между классами
T_line = np.linspace(20, 90, 500)
x_A, x_B = saturation(T_line)
T_e = T_line[np.argmax(x_A + x_B >= 1)]                         # температура эвтектики
borders = [(1 - x_A[(T_line >= T_e) & (x_A <= 1)], T_line[(T_line >= T_e) & (x_A <= 1)]),
           (x_B[(T_line >= T_e) & (x_B <= 1)], T_line[(T_line >= T_e) & (x_B <= 1)]),
           ([0, 1], [T_e, T_e])]

names = ["расплав", "расплав + A", "расплав + B", "кристаллы A + B"]
colors = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]
fig, ax = plt.subplots(figsize=(6.5, 4.4))
for k in range(4):
    part = mix[mix["класс"] == k]
    ax.scatter(part["x"], part["T"], s=8, color=colors[k], label=f"{k} - {names[k]}")
for x_border, T_border in borders:
    ax.plot(x_border, T_border, color="black", lw=1.2)
ax.set_xlim(0, 1)
ax.set_ylim(20, 90)
ax.set_xlabel("мольная доля нафталина")
ax.set_ylabel("температура, °C")
ax.legend(frameon=False, ncol=2, markerscale=2, loc="lower left", bbox_to_anchor=(0, 1.0))
plt.show()

In [ ]:
# Выборки и стандартизация признаков: те же шаги, что для поверхности энергии
X_all = mix[["x", "T"]].to_numpy()                   # признаки: состав и температура
y_all = mix["класс"].to_numpy()                      # ответы: класс 0, 1, 2 или 3
xT_train, xT_rest, y_train, y_rest = sklearn.model_selection.train_test_split(X_all, y_all, test_size=0.3, random_state=0)
xT_val, xT_test, y_val, y_test = sklearn.model_selection.train_test_split(xT_rest, y_rest, test_size=0.5, random_state=0)

scaler = sklearn.preprocessing.StandardScaler().fit(xT_train)
X_train, X_val, X_test = scaler.transform(xT_train), scaler.transform(xT_val), scaler.transform(xT_test)
print("обучающая:", X_train.shape, " валидационная:", X_val.shape, " тестовая:", X_test.shape)

In [ ]:
# Обучающая, валидационная и тестовая выборки с границами между классами
parts = [("обучающая", xT_train, y_train), 
         ("валидационная", xT_val, y_val), 
         ("тестовая", xT_test, y_test)]

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0), sharex=True, sharey=True)
for ax, (name, xT, y) in zip(axes, parts):
    for k in range(4):
        ax.scatter(xT[y == k, 0], xT[y == k, 1], s=8, color=colors[k], label=f"{k} - {names[k]}")
    for x_border, T_border in borders:
        ax.plot(x_border, T_border, color="black", lw=0.8, ls="--")          # настоящие границы между классами
    ax.set_title(f"{name} выборка: {len(y)} точек")
    ax.set_xlim(0, 1)
    ax.set_ylim(20, 90)
    ax.set_xlabel("мольная доля нафталина")
axes[0].set_ylabel("температура, °C")
axes[1].legend(frameon=False, ncol=4, markerscale=2, loc="upper center", bbox_to_anchor=(0.5, -0.18))
plt.show()

In [ ]:
# Модель 2-8-4 для классификации фазового состояния смеси
torch.manual_seed(0)               # начальные веса и порядок батчей воспроизводимы
clf = torch.nn.Sequential(
    torch.nn.Linear(2, 8),         # скрытый слой: 2 признака, 8 нейронов
    torch.nn.Sigmoid(),            # функция активации скрытого слоя: сигмоида
    torch.nn.Linear(8, 4))         # выходной слой: 4 нейрона, по одному на класс

In [ ]:
show(clf)

In [ ]:
# Обучающая выборка в виде тензоров и её разбиение на батчи
X_t = torch.tensor(X_train, dtype=torch.float32)                             # признаки обучающей выборки: тензор 280 × 2
y_t = torch.tensor(y_train)                                                  # ответы: номера классов, 280 целых чисел
batches = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_t, y_t),
                                      batch_size=20, shuffle=True)           # батчи по 20 объектов, порядок объектов случайный

In [ ]:
# Обучение: функция потерь - перекрёстная энтропия
optimizer = torch.optim.Adam(clf.parameters(), lr=0.05)                      # оптимизатор Adam и скорость обучения
n_epochs = 300                                                               # число эпох

loss_curve = []
for epoch in range(n_epochs):
    for X_batch, y_batch in batches:                                         # один шаг обучения на каждый батч
        optimizer.zero_grad()                                                # обнуление производных прошлого шага
        loss = torch.nn.functional.cross_entropy(clf(X_batch), y_batch)      # перекрёстная энтропия на батче: softmax вычисляется внутри
        loss.backward()                                                      # обратный проход: производные по весам и смещениям
        optimizer.step()                                                     # обновление весов и смещений
    with torch.no_grad():
        loss_curve.append(torch.nn.functional.cross_entropy(clf(X_t), y_t).item())   # перекрёстная энтропия на всей обучающей выборке после эпохи

In [ ]:
show(clf, verbose=True)

In [ ]:
# Вероятности классов и доля верных предсказаний на обучающей и валидационной выборках
def predict_proba(X):  # вероятности классов для массива NumPy: softmax от выходов модели
    with torch.no_grad():
        return torch.softmax(clf(torch.tensor(X, dtype=torch.float32)), dim=1).numpy()

print(f"доля верных предсказаний, обучающая выборка: {np.mean(predict_proba(X_train).argmax(axis=1) == y_train):.2f}")
print(f"доля верных предсказаний, валидационная выборка: {np.mean(predict_proba(X_val).argmax(axis=1) == y_val):.2f}")

In [ ]:
# Перекрёстная энтропия по эпохам и предсказанный класс на сетке значений состава и температуры
x_grid, T_grid = np.meshgrid(np.linspace(0, 1, 200), np.linspace(20, 90, 200))
grid = scaler.transform(np.column_stack([x_grid.ravel(), T_grid.ravel()]))
classes = predict_proba(grid).argmax(axis=1)                                 # предсказанный класс: тот, у которого вероятность наибольшая

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
axes[0].semilogy(loss_curve, color="#1f77b4", lw=0.9)
axes[0].set_title("обучающая выборка")
axes[0].set_xlabel("эпоха")
axes[0].set_ylabel("перекрёстная энтропия")

axes[1].contourf(x_grid, T_grid, classes.reshape(x_grid.shape), levels=[-0.5, 0.5, 1.5, 2.5, 3.5],
                 colors=["#f4c7c7", "#fde3c8", "#d5ecd0", "#cfe1f2"])        # области предсказанных классов
axes[1].scatter(xT_train[:, 0], xT_train[:, 1], s=6, c=[colors[k] for k in y_train])         # объекты обучающей выборки
for x_border, T_border in borders:
    axes[1].plot(x_border, T_border, color="black", lw=0.8, ls="--")         # настоящие границы между классами
axes[1].set_title("предсказанный класс")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(20, 90)
axes[1].set_xlabel("мольная доля нафталина")
axes[1].set_ylabel("температура, °C")
plt.show()

## 3.6 Свёрточные нейронные сети

**Свёрточная нейронная сеть** (convolutional neural network, CNN) - модель, в которой первые слои не связывают каждый нейрон со всеми признаками, а применяют **свёртку** (convolution): небольшой набор весов, **фильтр**, скользит вдоль данных и в каждом положении вычисляет взвешенную сумму нескольких соседних значений:

$$h_i = \varphi\!\left(\sum_{k=1}^{K} w_k \, x_{i+k-1} + b\right)$$

Здесь $x$ - значения признаков, идущие по порядку (точки спектра, пиксели изображения), $K$ - ширина фильтра. У свёрточного слоя два свойства:

- **локальность**: выход $h_i$ зависит только от $K$ соседних значений, а не от всех признаков сразу;
- **общие веса**: один и тот же фильтр применяется во всех положениях, поэтому весов мало, и фрагмент распознаётся независимо от того, в каком месте он находится.

Дальше стоят полносвязные слои. Веса фильтров не задаются заранее, а обучаются вместе с остальными весами модели методом обратного распространения ошибки.

Свёрточные модели хорошо работают там, где признаки упорядочены и один и тот же фрагмент может встретиться в разных местах: изображения, спектры, последовательности.

In [ ]:
# Набор данных: спектры с одной полосой и с двумя близкими полосами в случайном месте шкалы
wavelengths = np.linspace(400, 700, 301)             # длины волн, нм: шаг 1 нм

def band(center):                                    # одна полоса поглощения: гауссова кривая, σ = 10 нм
    return np.exp(-0.5 * ((wavelengths - center) / 10) ** 2)

def make_spectra(n, rng):                            # n спектров и их классы: 0 - одна полоса, 1 - две полосы
    y = rng.integers(0, 2, n)                                            # класс каждого спектра
    center = rng.uniform(450, 650, n)                                    # положение полос: случайное место шкалы
    X = np.array([band(c) if k == 0 else band(c - 15) + band(c + 15) for k, c in zip(y, center)])
    X = X * rng.uniform(0.5, 1.5, (n, 1))                                # случайная концентрация
    return X + 0.03 * rng.normal(size=X.shape), y                        # шум измерения

rng = np.random.default_rng(0)
X_train, y_train = make_spectra(60, rng)             # обучающая выборка: 60 спектров
X_val, y_val = make_spectra(600, rng)                # валидационная выборка: 600 спектров
print("обучающая:", X_train.shape, " валидационная:", X_val.shape)

In [ ]:
# Пять спектров каждого класса из обучающей выборки
names = ["одна полоса", "две полосы"]
colors = ["#1f77b4", "#ff7f0e"]
fig, axes = plt.subplots(2, 5, figsize=(12.5, 4.0), sharex=True, sharey=True)
for k in range(2):
    for ax, x in zip(axes[k], X_train[y_train == k]):                    # первые пять спектров класса k
        ax.plot(wavelengths, x, color=colors[k], lw=0.8)
    axes[k, 2].set_title(f"класс {k}: {names[k]}")
    axes[k, 0].set_ylabel("поглощение, усл. ед.")
for ax in axes[1]:
    ax.set_xlabel("длина волны, нм")
plt.tight_layout()
plt.show()

In [ ]:
# Выборки в виде тензоров и функции для двух моделей этого раздела
X_t = torch.tensor(X_train, dtype=torch.float32)                             # признаки обучающей выборки: тензор 60 × 301
y_t = torch.tensor(y_train)                                                  # ответы: номера классов, 60 целых чисел
X_v = torch.tensor(X_val, dtype=torch.float32)                               # признаки валидационной выборки: тензор 600 × 301
y_v = torch.tensor(y_val)

def train(net, n_epochs=300):                                                # обучение по всей обучающей выборке
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)                  # оптимизатор Adam и скорость обучения
    curve = []                                                               # перекрёстная энтропия после каждой эпохи
    for epoch in range(n_epochs):
        optimizer.zero_grad()                                                # обнуление производных прошлого шага
        loss = torch.nn.functional.cross_entropy(net(X_t), y_t)              # перекрёстная энтропия: softmax вычисляется внутри
        loss.backward()                                                      # обратный проход: производные по весам и смещениям
        optimizer.step()                                                     # обновление весов и смещений
        with torch.no_grad():
            loss_val = torch.nn.functional.cross_entropy(net(X_v), y_v)      # перекрёстная энтропия на валидационной выборке
        curve.append([loss.item(), loss_val.item()])
    return np.array(curve)                                                   # столбцы: обучающая и валидационная выборки

def accuracy(net, X, y):                                                     # доля верных предсказаний модели на выборке
    with torch.no_grad():
        return np.mean(net(torch.tensor(X, dtype=torch.float32)).argmax(dim=1).numpy() == y)

In [ ]:
# Обычная модель прямого распространения 301-32-2
torch.manual_seed(0)               # начальные веса воспроизводимы
mlp = torch.nn.Sequential(
    torch.nn.Linear(301, 32),      # скрытый слой: 301 признак, 32 нейрона
    torch.nn.ReLU(),               # функция активации скрытого слоя: ReLU
    torch.nn.Linear(32, 2))        # выходной слой: 2 нейрона, по одному на класс

show(mlp)

In [ ]:
mlp_curve = train(mlp)

In [ ]:
# Свёрточная модель: 8 фильтров шириной 31 точка, наибольший отклик каждого фильтра, выходной слой
torch.manual_seed(0)                           # начальные веса воспроизводимы
cnn = torch.nn.Sequential(
    torch.nn.Unflatten(1, (1, 301)),           # спектр как один канал из 301 точки: тензор n × 1 × 301
    torch.nn.Conv1d(1, 8, kernel_size=31),     # свёрточный слой: 8 фильтров по 31 весу
    torch.nn.ReLU(),                           # функция активации: ReLU
    torch.nn.AdaptiveMaxPool1d(1),             # пулинг: наибольший отклик каждого фильтра вдоль спектра
    torch.nn.Flatten(),                        # 8 откликов каждого спектра в одну строку: тензор n × 8
    torch.nn.Linear(8, 2))                     # выходной слой: 2 нейрона, по одному на класс

show(cnn)

Путь одного спектра через свёрточную модель:

| Слой | Что делается | Сколько чисел |
|---|---|---|
| вход | спектр | 301 |
| **свёртка** | окно из 31 точки сдвигается вдоль спектра на одну точку за шаг, всего $301 - 31 + 1 = 271$ положение. В каждом положении каждый из 8 фильтров вычисляет взвешенную сумму 31 точки и прибавляет смещение | 8 × 271 |
| функция активации ReLU | отрицательные отклики заменяются нулём | 8 × 271 |
| **пулинг** | Для каждого фильтра (h1-h8) выбирается наибольший из всей выборки (271) | 8 |
| выходной слой | по 8 числам вычисляются выходы $\hat{y}_1$ и $\hat{y}_2$, по одному на класс | 2 |

In [ ]:
cnn_curve = train(cnn)

In [ ]:
show(cnn)

In [ ]:
# Фильтры, найденные при обучении: веса свёрточного слоя
filters = cnn[1].weight.detach().numpy()[:, 0, :]                            # 8 фильтров по 31 весу
fig, axes = plt.subplots(2, 4, figsize=(12.5, 4.0), sharex=True, sharey=True)
for j, ax in enumerate(axes.ravel()):
    ax.axhline(0, color="grey", lw=0.6)
    ax.plot(filters[j], color="#1f77b4", lw=1.2)
    ax.set_title(f"фильтр {j + 1}")
for ax in axes[1]:
    ax.set_xlabel("номер веса")
for ax in axes[:, 0]:
    ax.set_ylabel("вес")
plt.tight_layout()
plt.show()


In [ ]:
# Наибольший отклик каждого фильтра на два спектра одинаковой высоты без шума
clean = np.array([band(550), band(535) + band(565)])                         # два спектра: одна полоса и две полосы
with torch.no_grad():
    response = cnn[:5](torch.tensor(clean, dtype=torch.float32)).numpy()     # слои модели до пулинга включительно: 2 × 8 чисел

position = np.arange(1, 9)                                                   # номера фильтров
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.2))
for k in range(2):
    axes[0].plot(wavelengths, clean[k], color=colors[k], lw=1.2, label=names[k])
    axes[1].bar(position + 0.4 * k - 0.2, response[k], width=0.4, color=colors[k], label=names[k])
axes[0].set_xlabel("длина волны, нм")
axes[0].set_ylabel("поглощение, усл. ед.")
axes[0].legend(frameon=False)
axes[1].set_xticks(position)
axes[1].set_xlabel("номер фильтра")
axes[1].set_ylabel("наибольший отклик")
plt.tight_layout()
plt.show()

In [ ]:
# Сравнение полносвязной и свёрточной моделей
models = [("Полносвязная модель", mlp, mlp_curve, "#2ca02c"), ("Свёрточная модель", cnn, cnn_curve, "#d62728")]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.4))
for i, (name, net, curve, color) in enumerate(models):
    axes[0].semilogy(curve[:, 0], color=color, lw=1.2, label=f"{name}, обучающая выборка")
    axes[0].semilogy(curve[:, 1], color=color, lw=1.2, ls="--", label=f"{name}, валидационная выборка")
    bars = axes[1].bar(np.arange(2) + 0.4 * i - 0.2, [accuracy(net, X_train, y_train), accuracy(net, X_val, y_val)],
                       width=0.4, color=color, label=name)
    axes[1].bar_label(bars, fmt="%.2f")
axes[0].set_xlabel("эпоха")
axes[0].set_ylabel("перекрёстная энтропия")
axes[0].legend(frameon=False, fontsize=8)
axes[1].set_xticks(np.arange(2), ["обучающая выборка", "валидационная выборка"])
axes[1].set_ylabel("доля верных предсказаний")
axes[1].set_ylim(0, 1.25)
axes[1].legend(frameon=False, ncol=2, loc="upper center")
plt.tight_layout()
plt.show()

# 🧩 Глава 4. Графовые нейронные сети (GNN)

**Графовые нейронные сети** (graph neural networks, GNN) - нейронные сети, которые принимают на вход не вектор признаков, а **граф** в виде набора вершин и рёбер между ними. Это особенно удобно для химии, поскольку молекула естественно описывается графом: атомы - вершины графа, химические связи - рёбра.

## 4.1 Представление в виде графа

**Граф** (graph) - набор **вершин** (nodes) и **рёбер** (edges), которые соединяют пары вершин. Граф описывает не сами объекты, а то, какие из них связаны между собой. Вершины графа можно пронумеровать, но сами номера смысла не несут: если вершины перенумеровать, граф останется тем же.

In [ ]:
# Граф из 6 вершин
G = nx.Graph()
G.add_edges_from([(0, 1), (0, 2), (0, 3), (0, 4), (1, 5)])          # рёбра: пары номеров соединённых вершин
print("вершины:", list(G.nodes))
print("рёбра:", list(G.edges))
print("соседи вершины 0:", list(G.neighbors(0)))
print("степени вершин:", dict(G.degree))
print("расстояние между вершинами 2 и 5:", nx.shortest_path_length(G, 2, 5))

fig, ax = plt.subplots(figsize=(4.5, 3))
nx.draw(G, nx.kamada_kawai_layout(G), ax=ax, with_labels=True, node_size=500, node_color="white", edgecolors="black")
plt.show()

**Матрица смежности** (adjacency matrix) $A$ - таблица размера $n \times n$: $A_{ij} = 1$, если вершины $i$ и $j$ соединены ребром, и $0$, если нет. Вершины, соединённые ребром, называют соседями. При перенумерации вершин строки и столбцы $A$ переставляются, хотя граф остаётся тем же, поэтому модель не должна зависеть от нумерации.

In [ ]:
# Запись графа для модели: матрица смежности A и вектор признаков вершин X
A = torch.tensor(nx.to_numpy_array(G, nodelist=range(6)), dtype=torch.float32)   # матрица смежности: тензор 6 × 6, строки и столбцы по номерам вершин
X = torch.ones(6, 1)                                                 # признаки вершин: одно число на вершину, у всех вершин 1

# Тот же граф с другой нумерацией вершин
new_number = {0: 3, 1: 5, 2: 0, 3: 4, 4: 1, 5: 2}                    # перенумерация: старый номер вершины -> новый
G_new = nx.relabel_nodes(G, new_number)                              # тот же граф с новыми номерами вершин
position = nx.kamada_kawai_layout(G)                                 # положение вершин на рисунке
position_new = {new_number[v]: xy for v, xy in position.items()}     # те же положения для новых номеров

fig, axes = plt.subplots(2, 3, figsize=(9.5, 6.0), gridspec_kw={"width_ratios": [3, 2.2, 0.5]})
for row, (graph, pos) in enumerate([(G, position), (G_new, position_new)]):
    A_row = nx.to_numpy_array(graph, nodelist=range(6))              # матрица смежности для этой нумерации
    nx.draw(graph, pos, ax=axes[row, 0], with_labels=True, node_size=500, node_color="white", edgecolors="black")
    axes[row, 1].imshow(A_row, cmap="Greys", vmin=0, vmax=3)         # серая клетка - ребро есть, белая - ребра нет
    axes[row, 2].imshow(X, cmap="Greys", vmin=0, vmax=3)
    for i in range(6):
        axes[row, 2].text(0, i, f"{X[i, 0]:.0f}", ha="center", va="center")
        for j in range(6):
            axes[row, 1].text(j, i, f"{A_row[i, j]:.0f}", ha="center", va="center")
    axes[row, 1].set_xticks(range(6))
    axes[row, 1].set_yticks(range(6))
    axes[row, 1].set_xlabel("номер вершины")
    axes[row, 1].set_ylabel("номер вершины")
    axes[row, 1].set_title("матрица смежности A")
    axes[row, 2].set_xticks([])
    axes[row, 2].set_yticks(range(6))
    axes[row, 2].set_title("X")
plt.tight_layout()
plt.show()

$X$ - признаки вершин. В этом примере у каждой вершины один признак, поэтому $X$ - вектор из $n$ чисел, по одному на вершину. Если признаков несколько, $X$ - матрица размера $n \times m$: строка - вершина, столбец - признак.

In [ ]:
# Признаки вершин: один признак на вершину и три признака на вершину
X1 = torch.tensor([[0.5], [0.3], [0.2], [0.2], [0.6], [0.7]])        # один признак на вершину: тензор 6 × 1
X3 = torch.tensor([[0.5, 0.6, 0.5],                                  # три признака на вершину: тензор 6 × 3, строка - вершина
                   [0.7, 0.6, 0.2],
                   [0.6, 0.2, 0.6],
                   [0.3, 0.6, 0.5],
                   [0.3, 0.4, 0.2],
                   [0.3, 0.5, 0.5]])

position = nx.kamada_kawai_layout(G)                                 # положение вершин на рисунке
fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.4), gridspec_kw={"width_ratios": [3, 0.7, 3, 1.4]})
for k, values in enumerate([X1, X3]):
    m = values.shape[1]                                              # число признаков
    nx.draw(G, position, ax=axes[2 * k], with_labels=True, node_size=500, node_color="white", edgecolors="black")
    for i in range(6):                                               # красным рядом с вершиной - её признаки, столбиком
        axes[2 * k].text(position[i][0] + 0.1, position[i][1] + 0.08, "\n".join(f"{v:.1f}" for v in values[i]), color="#d62728",
                         fontsize=9, linespacing=0.9, va="bottom", bbox={"facecolor": "white", "edgecolor": "none", "pad": 1})
    axes[2 * k].margins(0.25)
    ax = axes[2 * k + 1]
    ax.imshow(values, cmap="Greys", vmin=0, vmax=2)
    for i in range(6):
        for j in range(m):
            ax.text(j, i, f"{values[i, j]:.1f}", ha="center", va="center", color="#d62728")
    ax.set_xticks(range(m), range(1, m + 1))
    ax.set_yticks(range(6))
    ax.set_xlabel("номер признака")
    ax.set_ylabel("номер вершины")
    ax.set_title(f"X: 6 × {m}")
plt.tight_layout()
plt.show()

## 4.2 Графовый слой

**Графовый слой** (graph layer) вычисляет для каждой вершины новое число по ней самой и по её соседям. Для вершины $i$ это два действия:

1. складываются числа самой вершины и всех её соседей: $s_i = x_i + \sum_{j} x_j$, где $j$ - соседи вершины $i$;
2. к сумме применяется обычный нейрон: $h_i = \varphi(w s_i + b)$.

Вес $w$ и смещение $b$ одни и те же для всех вершин, как у фильтра свёрточного слоя, который одинаков во всех положениях окна. Поэтому слой применим к графу любого размера и не зависит от нумерации вершин.

Сумма по соседям для всех вершин сразу записывается через матрицу смежности: $(A + I)X$, где $I$ - единичная матрица, она добавляет к соседям саму вершину. Весь слой:

$$H = \varphi\big((A + I)\,X\,w + b\big)$$

После одного слоя число вершины зависит от её соседей, после двух - ещё и от соседей соседей: каждый слой расширяет окрестность вершины на одно ребро.

In [ ]:
# Графовый слой как модель: к сумме признаков вершины и её соседей применяется нейрон с сигмоидой
layer1 = torch.nn.Sequential(
    torch.nn.Linear(1, 1),                           # один признак на входе, одно число на выходе: один вес и одно смещение на все вершины
    torch.nn.Sigmoid())                              # функция активации: сигмоида
with torch.no_grad():                                # вес 0.5 и смещение -0.5 заданы вручную
    layer1[0].weight.fill_(0.5)
    layer1[0].bias.fill_(-0.5)

def graph_layer(layer, A, X):                               # графовый слой: матрица смежности A и признаки вершин X -> новые числа вершин
    I = torch.eye(len(A))                            # единичная матрица: вершина учитывает и свой признак
    return layer((A + I) @ X)                        # сумма по вершине и её соседям, затем нейрон: φ(вес · сумма + смещение)

with torch.no_grad():
    H = graph_layer(layer1, A, X1)                           # новые числа вершин: тензор 6 × 1

S = (A + torch.eye(6)) @ X1                          # суммы признаков вершины и её соседей: тензор 6 × 1
show(layer1, input=S[0], verbose=True)                # нейрон слоя для вершины 0: на входе сумма, на выходе новое число вершины

In [ ]:
# Графовый слой: расчёт для двух вершин и результат для всех вершин
w, b = layer1[0].weight.item(), layer1[0].bias.item()                # вес и смещение берутся из модели layer1
with torch.no_grad():
    H = graph_layer(layer1, A, X1)                                   # новые числа всех вершин считает модель: тензор 6 × 1

position = nx.kamada_kawai_layout(G)                                 # положение вершин на рисунке
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))
for ax, target in zip(axes[:2], [1, 2]):                             # две вершины с разным числом соседей
    neighbors = list(G.neighbors(target))
    node_colors = ["#f1948a" if i == target else "#fad7a0" if i in neighbors else "white" for i in range(6)]
    nx.draw(G, position, ax=ax, with_labels=True, node_size=500, node_color=node_colors, edgecolors="black", edge_color="lightgrey")
    for j in neighbors:                                              # стрелки от соседей к вершине
        ax.annotate("", xy=position[target], xytext=position[j],
                    arrowprops={"arrowstyle": "->", "color": "#ff7f0e", "lw": 2, "shrinkA": 12, "shrinkB": 12})
    for i in range(6):                                               # рядом с вершиной - её признак
        ax.text(position[i][0] + 0.1, position[i][1] + 0.08, f"{X1[i, 0]:.1f}", fontsize=9, va="bottom", color="#d62728")
    terms = [target] + neighbors                                     # сама вершина и её соседи
    s = X1[terms, 0].sum()                                           # сумма их признаков
    ax.set_title(f"вершина {target}\n" + " + ".join(f"{X1[i, 0]:.1f}" for i in terms) + f" = {s:.1f}\n"
                 f"φ({w:.3g} · {s:.1f} {'+' if b >= 0 else '-'} {abs(b):.3g}) = {H[target, 0]:.4f}", fontsize=11)
    ax.margins(0.15)

nx.draw(G, position, ax=axes[2], with_labels=True, node_size=500, node_color="white", edgecolors="black")
for i in range(6):                                                   # рядом с вершиной - её новое число
    axes[2].text(position[i][0] + 0.1, position[i][1] + 0.08, f"{H[i, 0]:.4f}", fontsize=9, va="bottom", color="#d62728")
axes[2].set_title("После графового слоя", fontsize=11)
axes[2].margins(0.15)
plt.tight_layout()
plt.show()

In [ ]:
# Графовый слой с несколькими признаками как модель: у вершины 3 числа на входе и 2 числа на выходе
torch.manual_seed(0)                                 # начальные веса воспроизводимы
layer3 = torch.nn.Sequential(
    torch.nn.Linear(3, 2),                           # 3 признака на входе, 2 числа на выходе: 6 весов и 2 смещения, общие для всех вершин
    torch.nn.Sigmoid())                              # функция активации: сигмоида

with torch.no_grad():
    H3 = graph_layer(layer3, A, X3)                  # та же функция с другим слоем: новые векторы вершин, тензор 6 × 2

S3 = (A + torch.eye(6)) @ X3                         # суммы по вершине и её соседям: тензор 6 × 3
show(layer3, input=S3[0], verbose=True)              # нейроны слоя для вершины 0: на входе три суммы, на выходе два новых числа

In [ ]:
# Признаки вершин до и после графового слоя: было 3 числа у вершины, стало 2
position = nx.kamada_kawai_layout(G)                                 # положение вершин на рисунке
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
for ax, values, title, fmt in [(axes[0], X3, "до: 3 числа у вершины", "{:.1f}"), (axes[1], H3, "после: 2 числа у вершины", "{:.2f}")]:
    nx.draw(G, position, ax=ax, with_labels=True, node_size=500, node_color="white", edgecolors="black")
    for i in range(6):                                               # красным рядом с вершиной - её числа, столбиком
        ax.text(position[i][0] + 0.1, position[i][1] + 0.08, "\n".join(fmt.format(v) for v in values[i]), color="#d62728",
                fontsize=9, linespacing=0.9, va="bottom", bbox={"facecolor": "white", "edgecolor": "none", "pad": 1})
    ax.set_title(title)
    ax.margins(0.25)
plt.tight_layout()
plt.show()

После графового слоя у каждой вершины есть свой вектор $h_i$. Чтобы перейти от вершин к свойству всего графа, эти вектора складывают:

$$h_\text{граф} = \sum_i h_i$$

Сумма не зависит от нумерации вершин и определена для графа с любым числом вершин. Дальше работает полносвязный слой: по сумме он вычисляет предсказание $\hat{y}$ - свойство графа. 

Вся модель состоит из трёх шагов: графовый слой, сумма по вершинам, полносвязный слой. Весов и смещений в ней столько же, сколько в этих двух слоях, и от размера графа их число не зависит. Поэтому одну и ту же модель можно применять к разным графам.

In [ ]:
# Векторы вершин после графового слоя: строка - вершина, столбец - число вектора
table = pd.DataFrame(H3.numpy(), columns=["число 1", "число 2"])
table.index.name = "вершина"
table.round(4)

In [ ]:
print("вектор графа:", H3.sum(dim=0))                # сумма по вершинам: тензор из 2 чисел

## 4.3 Признаки рёбер и передача сообщений

У ребра, как и у вершины, могут быть свои **признаки** (edge features). Графовый слой из раздела 4.2 их не использует: все соседи входят в сумму одинаково.

**Передача сообщений** (message passing) - обобщение графового слоя. Новый вектор вершины вычисляется в три действия:

1. каждый сосед $j$ отправляет вершине $i$ **сообщение** $m_{ij}$. Оно вычисляется по вектору соседа $h_j$ и признакам ребра $e_{ij}$
2. сообщения от всех соседей складываются
3. по собственному вектору вершины и сумме сообщений вычисляется новый вектор вершины

$$m_{ij} = \varphi\big(W\,[h_j,\ e_{ij}]\big), \qquad h_i' = \varphi\big(W_0 h_i + \sum_j m_{ij} + b\big)$$

Здесь $[h_j,\ e_{ij}]$ - вектор соседа и признаки ребра, записанные подряд.Веса $W$, $W_0$ и смещение $b$. Модель из таких слоёв называют **нейронной сетью передачи сообщений** (message passing neural network, MPNN).

In [ ]:
# Слой передачи сообщений как модель: сообщение вычисляется по признаку соседа и признаку ребра
class MPNNLayer(torch.nn.Module):
    """Слой передачи сообщений с признаками рёбер.
    Для каждой пары (i, j): сообщение m_ij = ReLU(W [h_j, e_ij])
    Новый вектор вершины: h_i = ReLU(сумма сообщений от соседей + b)
    """
    def __init__(self, n_node, n_edge, units):
        super().__init__()
        self.w_msg = torch.nn.Linear(n_node + n_edge, units, bias=False)     # веса сообщения: общие для всех рёбер
        self.w_self = torch.nn.Linear(n_node, units, bias=False)             # вес собственного вектора вершины
        self.b = torch.nn.Parameter(torch.zeros(units))                      # смещение: общее для всех вершин

    def forward(self, X, A, E):                      # X: n × n_node, A: n × n, E: n × n × n_edge
        n = len(X)
        X_j = X.unsqueeze(0).expand(n, n, -1)        # вектор соседа j для каждой пары (i, j): тензор n × n × n_node
        msg = torch.relu(self.w_msg(torch.cat([X_j, E], dim=-1)))            # сообщения: признаки соседа и ребра подряд, затем веса, тензор n × n × units
        msg = A.unsqueeze(-1) * msg                  # остаются сообщения только по существующим рёбрам
        return torch.relu(self.w_self(X) + msg.sum(dim=1) + self.b)          # новый вектор вершины: h_i' = ReLU(W_0 h_i + сумма сообщений от соседей + b)

In [ ]:
edge_values = {(0, 1): 2.0, (0, 2): 1.0, (0, 3): 0.5, (0, 4): 0.5, (1, 5): 3.0}      # признак каждого ребра: одно число
E = torch.zeros(6, 6, 1)                             # признаки рёбер: тензор 6 × 6 × 1
for (i, j), value in edge_values.items():
    E[i, j, 0] = E[j, i, 0] = value                  # у ребра один признак в обе стороны

layer_mp = MPNNLayer(1, 1, 1)                        # 1 признак вершины, 1 признак ребра, 1 число на выходе
with torch.no_grad():                                # веса сообщения заданы вручную: 1 для признака соседа, 0.5 для признака ребра
    layer_mp.w_msg.weight.copy_(torch.tensor([[1.0, 0.5]]))
    layer_mp.w_self.weight.fill_(1.0)
    H_mp = layer_mp(X1, A, E)                        # новые числа вершин: тензор 6 × 1

position = nx.kamada_kawai_layout(G)                 # положение вершин на рисунке
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
for ax, values, title in [(axes[0], X1, "признаки вершин и рёбер"), (axes[1], H_mp, "после слоя передачи сообщений")]:
    nx.draw(G, position, ax=ax, with_labels=True, node_size=500, node_color="white", edgecolors="black",
            width=[1.5 * edge_values[tuple(sorted(edge))] for edge in G.edges])          # толщина ребра - его признак
    nx.draw_networkx_edge_labels(G, position, ax=ax, edge_labels=edge_values, font_color="#1f77b4", font_size=9)   # синим - признаки рёбер: слой их не меняет
    for i in range(6):                               # красным рядом с вершиной - её число
        ax.text(position[i][0] + 0.1, position[i][1] + 0.08, f"{values[i, 0]:.2f}", fontsize=9, va="bottom", color="#d62728")
    ax.set_title(title)
    ax.margins(0.15)
plt.tight_layout()
plt.show()

In [ ]:
show(layer_mp, input=(X1, A, E), verbose=True)

# 📌 Дополнительные материалы

**Учебники и курсы**:
* [Учебник по машинному обучению Яндекс](https://education.yandex.ru/handbook/ml) - на русском, от линейных моделей до нейронных сетей
* [Deep Learning](https://www.deeplearningbook.org) (Goodfellow, Bengio, Courville) - учебник в открытом доступе; главы 6-9: сети прямого распространения, обучение, свёрточные сети
* [3Blue1Brown: Neural networks](https://www.3blue1brown.com/topics/neural-networks) - видео с наглядным объяснением градиентного спуска и обратного распространения ошибки

**Библиотеки**:
* [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html) - описание методов с примерами; к главе 2 относятся разделы [Clustering](https://scikit-learn.org/stable/modules/clustering.html) и [Decomposition](https://scikit-learn.org/stable/modules/decomposition.html)
* [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) - официальные уроки: тензоры, автоматическое дифференцирование, обучение моделей

**Понижение размерности**:
* [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/) - интерактивная статья о том, как читать карты t-SNE и в чём они могут обмануть
* [UMAP](https://umap-learn.readthedocs.io/en/latest/) - документация с объяснением метода

**Свёрточные и графовые нейронные сети**:
* [CNN Explainer](https://poloclub.github.io/cnn-explainer/) - интерактивная схема свёрточной модели: фильтры, свёртка, пулинг
* [A Gentle Introduction to Graph Neural Networks](https://distill.pub/2021/gnn-intro/) - интерактивное введение в графовые нейронные сети
* [Kipf, Welling, 2017](https://arxiv.org/abs/1609.02907) - графовый слой; [Gilmer et al., 2017](https://arxiv.org/abs/1704.01212) - передача сообщений для предсказания свойств молекул
* [PyTorch Geometric: Introduction by Example](https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html) - графовые модели на PyTorch